In [1]:
import os
import time
import traceback
import hashlib

import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

BLOCKSIZE = 2
WM_SIZE = 64
q = 8.0
Q_MARGIN = 0.026
R_MARGIN = 1.95
ARNOLD_ITER = 10
PRIVATE_KEY = "KB123"
Q_TAU = 0.5
FLAG_Q = 0
FLAG_R = 1
FLAG_SKIP = 2
EPS_SCORE = 1e-9
EPS_CONF = 1e-12

ATTACK_AWARE = True
NLM_ALWAYS_COMPARE = True
CONF_THRESH = 0.10

NLM_H1 = 4
NLM_TEMPLATE_WIN1 = 3
NLM_SEARCH_WIN1 = 11

NLM_H2 = 8
NLM_TEMPLATE_WIN2 = 5
NLM_SEARCH_WIN2 = 17

DEBUG = True
DEBUG_BLOCK_PRINT_LIMIT = 5


def _debug(msg):
    if DEBUG:
        print(msg)


def _flag_stats(flags):
    q_used = sum(1 for f in flags if int(f) == FLAG_Q)
    r_used = sum(1 for f in flags if int(f) == FLAG_R)
    skip_used = sum(1 for f in flags if int(f) == FLAG_SKIP)
    return q_used, r_used, skip_used


def _highest_odd_repeat(total_blocks, payload_bits):
    if payload_bits <= 0:
        raise ValueError("payload_bits must be positive.")

    full_repeat = total_blocks // payload_bits
    if full_repeat < 1:
        raise ValueError(
            f"Capacity insufficient: need at least {payload_bits} blocks, but only {total_blocks} available."
        )

    odd_repeat = full_repeat if (full_repeat % 2 == 1) else (full_repeat - 1)
    if odd_repeat < 1:
        odd_repeat = 1

    return odd_repeat


def load_wm_gray_rgba_safe(path, wm_size, bg=255):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read watermark file: {path}")

    if img.ndim == 3 and img.shape[2] == 4:
        bgr = img[..., :3].astype(np.float32)
        a = img[..., 3:4].astype(np.float32) / 255.0
        bg_bgr = np.full_like(bgr, float(bg), dtype=np.float32)
        comp = bgr * a + bg_bgr * (1.0 - a)
        gray = cv2.cvtColor(comp.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    elif img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    if gray.shape[0] != wm_size or gray.shape[1] != wm_size:
        raise ValueError(f"Watermark must be exactly {wm_size}x{wm_size}. Got {gray.shape[1]}x{gray.shape[0]}.")

    return gray


def arnold_transform(image, iterations):
    n = image.shape[0]
    result = image.copy()
    for _ in range(iterations):
        new = np.zeros_like(result)
        for x in range(n):
            for y in range(n):
                new[(x + y) % n, (x + 2 * y) % n] = result[x, y]
        result = new
    return result


def invert_arnold_transform(image, iterations):
    n = image.shape[0]
    result = image.copy()
    for _ in range(iterations):
        new = np.zeros_like(result)
        for x in range(n):
            for y in range(n):
                new[(2 * x - y) % n, (-x + y) % n] = result[x, y]
        result = new
    return result


def generate_block_indices(img_shape, block_size, total_bits_needed, key):
    h, w = img_shape[:2]
    blocks_per_row = w // block_size
    blocks_per_col = h // block_size
    total_blocks = blocks_per_row * blocks_per_col

    if total_bits_needed > total_blocks:
        raise ValueError(
            f"Capacity insufficient: Need {total_bits_needed} blocks, but image only has {total_blocks}."
        )

    key_bytes = hashlib.sha512(key.encode("utf-8")).digest()
    x0_int = int.from_bytes(key_bytes[:8], "big")
    x0 = x0_int / (2 ** 64)

    r = 3.99
    chaotic_seq = np.zeros(total_blocks, dtype=np.float64)
    x = x0

    for _ in range(100):
        x = r * x * (1 - x)

    for i in range(total_blocks):
        x = r * x * (1 - x)
        chaotic_seq[i] = x

    shuffled_indices = np.argsort(chaotic_seq)
    selected_indices = shuffled_indices[:total_bits_needed]

    indices = []
    for idx in selected_indices:
        row = (idx // blocks_per_row) * block_size
        col = (idx % blocks_per_row) * block_size
        indices.append((row, col))

    return indices


def _load_flags(flag_path):
    if not os.path.exists(flag_path):
        return []
    try:
        arr = np.loadtxt(flag_path, dtype=np.int32)
        if arr.size == 0:
            return []
        if np.ndim(arr) == 0:
            return [int(arr)]
        return [int(x) for x in arr.tolist()]
    except Exception:
        return []


def _safe_sign(x):
    return 1.0 if x >= 0 else -1.0


def _wrap_to_half_period(delta, period):
    return (delta + 0.5 * period) % period - 0.5 * period


def _qr_canonical(block: np.ndarray):
    Qm, Rm = np.linalg.qr(block)
    d = np.sign(np.diag(Rm))
    d[d == 0] = 1.0
    D = np.diag(d)
    Qm = Qm @ D
    Rm = D @ Rm
    return Qm, Rm


def _make_Q2_from_target(Qm: np.ndarray, c_target: float):
    c_target = float(np.clip(abs(c_target), 0.0, 0.999999))
    det_sign = 1.0 if np.linalg.det(Qm) >= 0 else -1.0
    c_sign = _safe_sign(float(Qm[1, 1]))
    s_sign = _safe_sign(float(Qm[0, 1]))
    c_new = c_sign * c_target
    s_new = s_sign * float(np.sqrt(max(0.0, 1.0 - c_target * c_target)))
    Q1 = np.array([[c_new, s_new], [-s_new, c_new]], dtype=np.float64)
    if det_sign < 0:
        Q1[:, 1] *= -1.0
    return Q1


def _llr_q_from_Q(Qm: np.ndarray) -> float:
    return abs(float(Qm[1, 1])) - float(Q_TAU)


def _llr_r_from_R(Rm: np.ndarray, q_eff: float) -> float:
    z = float(Rm[1, 1]) % float(q_eff)
    return z - 0.5 * float(q_eff)


def _norm_llr_q(llr_q: float) -> float:
    return float(llr_q) / max(0.5, EPS_CONF)


def _norm_llr_r(llr_r: float, q_eff: float) -> float:
    return float(llr_r) / max(0.5 * float(q_eff), EPS_CONF)


def _build_q_candidate(block: np.ndarray, bit: int, q_margin: float):
    Qm, Rm = _qr_canonical(block)
    a11 = abs(float(Qm[1, 1]))
    tgt0 = max(0.0, Q_TAU - 0.5 * q_margin)
    tgt1 = min(0.999999, Q_TAU + 0.5 * q_margin)

    need_Q_change = False
    if bit == 1:
        if a11 < tgt1:
            need_Q_change = True
    else:
        if a11 > tgt0:
            need_Q_change = True

    if need_Q_change:
        c_target = tgt1 if bit == 1 else tgt0
        Q1 = _make_Q2_from_target(Qm, c_target)
    else:
        Q1 = Qm

    block1 = Q1 @ Rm
    return block1


def _build_r_candidate(block: np.ndarray, bit: int, q_eff: float, r_margin: float):
    Qm, Rm = _qr_canonical(block)
    R2 = Rm.copy()
    r22 = float(R2[1, 1])
    z = r22 % q_eff

    safe_R = False
    if bit == 1:
        if z >= (0.5 * q_eff + r_margin) and z <= (q_eff - r_margin):
            safe_R = True
    else:
        if z <= (0.5 * q_eff - r_margin) and z >= r_margin:
            safe_R = True

    if not safe_R:
        target = 0.75 * q_eff if bit == 1 else 0.25 * q_eff
        delta = _wrap_to_half_period(target - z, q_eff)
        R2[1, 1] = r22 + delta

    block2 = Qm @ R2
    return block2


def _signed_margin_from_block(block: np.ndarray, mode: int, bit: int, q_eff: float):
    Qm, Rm = _qr_canonical(block)
    if mode == FLAG_Q:
        llr = _norm_llr_q(_llr_q_from_Q(Qm))
    else:
        llr = _norm_llr_r(_llr_r_from_R(Rm, q_eff), q_eff)
    return llr if bit == 1 else -llr


def _candidate_attack_score(block0: np.ndarray, block_cand: np.ndarray, mode: int, bit: int, q_eff: float):
    mse = float(np.mean((block0 - block_cand) ** 2))
    base = np.clip(np.rint(block_cand), 0, 255).astype(np.float64)

    variants = [base]
    variants.append(np.clip(base + 1.0, 0, 255))
    variants.append(np.clip(base - 1.0, 0, 255))

    blur = cv2.GaussianBlur(base.astype(np.float32), (3, 3), 0.6, borderType=cv2.BORDER_REFLECT_101)
    variants.append(np.clip(np.rint(blur), 0, 255).astype(np.float64))

    margins = []
    for v in variants:
        try:
            margins.append(_signed_margin_from_block(v, mode, bit, q_eff))
        except Exception:
            margins.append(-1.0)

    raw_margin = float(margins[0])
    avg_margin = float(np.mean(margins))
    worst_margin = float(np.min(margins))

    robust_score = 0.5 * raw_margin + 0.3 * avg_margin + 0.2 * worst_margin
    final_score = robust_score / (mse + EPS_SCORE)

    ok = (raw_margin >= 0.0 and avg_margin >= 0.0)

    return final_score, mse, raw_margin, avg_margin, worst_margin, ok


def _nlm_denoise_gray(gray_u8: np.ndarray, h: int, template_win: int, search_win: int):
    return cv2.fastNlMeansDenoising(
        src=gray_u8,
        dst=None,
        h=float(h),
        templateWindowSize=int(template_win),
        searchWindowSize=int(search_win)
    )


def embed(host_path, watermark_path, output_path, flag_path, arnold_iter=10):
    host_img_orig = cv2.imread(host_path)
    if host_img_orig is None:
        raise ValueError(f"Cannot read host image: {host_path}")

    host_channel_u8 = host_img_orig[:, :, 0]
    H_img, W_img = host_channel_u8.shape

    H0 = (H_img // BLOCKSIZE) * BLOCKSIZE
    W0 = (W_img // BLOCKSIZE) * BLOCKSIZE
    if H0 == 0 or W0 == 0:
        raise ValueError("Image too small for BLOCKSIZE constraint.")

    work = host_channel_u8[:H0, :W0].astype(np.float64)

    wm_gray = load_wm_gray_rgba_safe(watermark_path, WM_SIZE)
    wm_scrambled = arnold_transform(wm_gray, arnold_iter)
    wm_bits = (wm_scrambled.flatten() > 127).astype(np.uint8)
    L = WM_SIZE * WM_SIZE

    total_blocks = (H0 // BLOCKSIZE) * (W0 // BLOCKSIZE)
    if total_blocks < L:
        raise ValueError(f"Capacity insufficient: need {L} blocks, but only {total_blocks} available.")

    odd_repeat = _highest_odd_repeat(total_blocks, L)
    selected_blocks = L * odd_repeat
    blocks = generate_block_indices(work.shape, BLOCKSIZE, selected_blocks, PRIVATE_KEY)

    q_eff = max(float(q), 1e-6)
    q_margin = max(0.0, float(Q_MARGIN))
    q_margin = min(q_margin, 2.0 * min(Q_TAU, 1.0 - Q_TAU) - 1e-6)
    r_margin = max(0.0, float(R_MARGIN))
    r_margin = min(r_margin, 0.49 * q_eff)

    _debug(f"[EMBED] host='{os.path.basename(host_path)}'")
    _debug(f"[EMBED] original_size={W_img}x{H_img}, working_size={W0}x{H0}, channel=B")
    _debug(f"[EMBED] payload_bits={L}, total_blocks={total_blocks}, selected_blocks={len(blocks)}")
    _debug(f"[EMBED] full_repeat={total_blocks // L}, odd_repeat={odd_repeat}")
    _debug(f"[EMBED] repeat_factor={len(blocks) / float(L):.4f}x")
    _debug(f"[EMBED] q_eff={q_eff:.6f}, q_margin={q_margin:.6f}, r_margin={r_margin:.6f}")
    _debug(f"[EMBED] wm_bits_ones={int(np.sum(wm_bits))}, wm_bits_zeros={int(L - np.sum(wm_bits))}")

    watermarked = work.copy()
    flag_list = []

    for idx, (i, j) in enumerate(blocks):
        bit = int(wm_bits[idx % L])
        block0 = watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE].copy()

        try:
            block_q = _build_q_candidate(block0, bit, q_margin)
            score_q, mse_q, raw_q, avg_q, worst_q, ok_q = _candidate_attack_score(
                block0, block_q, FLAG_Q, bit, q_eff
            )

            block_r = _build_r_candidate(block0, bit, q_eff, r_margin)
            score_r, mse_r, raw_r, avg_r, worst_r, ok_r = _candidate_attack_score(
                block0, block_r, FLAG_R, bit, q_eff
            )

            if idx < DEBUG_BLOCK_PRINT_LIMIT:
                _debug(
                    f"[EMBED][blk={idx}] pos=({i},{j}) bit={bit} | "
                    f"Q: score={score_q:.6f}, mse={mse_q:.6f}, raw={raw_q:.6f}, avg={avg_q:.6f}, worst={worst_q:.6f}, ok={ok_q} | "
                    f"R: score={score_r:.6f}, mse={mse_r:.6f}, raw={raw_r:.6f}, avg={avg_r:.6f}, worst={worst_r:.6f}, ok={ok_r}"
                )

            if ok_q and ok_r:
                if score_q > score_r or (abs(score_q - score_r) <= 1e-12 and mse_q <= mse_r):
                    watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_q
                    flag_list.append(FLAG_Q)
                else:
                    watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_r
                    flag_list.append(FLAG_R)
            elif ok_q:
                watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_q
                flag_list.append(FLAG_Q)
            elif ok_r:
                watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_r
                flag_list.append(FLAG_R)
            else:
                if score_q > score_r or (abs(score_q - score_r) <= 1e-12 and mse_q <= mse_r):
                    watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_q
                    flag_list.append(FLAG_Q)
                else:
                    watermarked[i:i + BLOCKSIZE, j:j + BLOCKSIZE] = block_r
                    flag_list.append(FLAG_R)

        except Exception as e:
            if idx < DEBUG_BLOCK_PRINT_LIMIT:
                _debug(f"[EMBED][blk={idx}] pos=({i},{j}) bit={bit} -> EXCEPTION -> SKIP | {e}")
            flag_list.append(FLAG_SKIP)
            continue

    out_channel = host_channel_u8.astype(np.float64).copy()
    out_channel[:H0, :W0] = np.clip(np.rint(watermarked), 0, 255)

    watermarked_img_final = host_img_orig.copy()
    watermarked_img_final[:, :, 0] = out_channel.astype(np.uint8)

    cv2.imwrite(output_path, watermarked_img_final)
    np.savetxt(flag_path, np.array(flag_list, dtype=np.uint8), fmt="%d")

    q_used, r_used, skip_used = _flag_stats(flag_list)
    total_used = len(flag_list)

    _debug(f"[EMBED] output='{output_path}'")
    _debug(f"[EMBED] flag_path='{flag_path}'")
    _debug(f"[EMBED] Q used={q_used}, R used={r_used}, SKIP used={skip_used}, total_flags={total_used}")

    if total_used > 0:
        _debug(
            f"[EMBED] ratios -> "
            f"Q={q_used / total_used:.4f}, "
            f"R={r_used / total_used:.4f}, "
            f"SKIP={skip_used / total_used:.4f}"
        )

    return watermarked_img_final


def _extract_from_work(work: np.ndarray, flags, blocks, q_eff, original_wm_shape, arnold_iter):
    L = original_wm_shape[0] * original_wm_shape[1]
    n = min(len(blocks), len(flags))

    q_used, r_used, skip_used = _flag_stats(flags[:n])
    _debug(
        f"[EXTRACT-INTERNAL] blocks_considered={n}, "
        f"Q={q_used}, R={r_used}, SKIP={skip_used}, payload_bits={L}"
    )

    sum_score = np.zeros(L, dtype=np.float64)
    sum_abs = np.zeros(L, dtype=np.float64)
    sum_count = np.zeros(L, dtype=np.float64)

    for idx in range(n):
        i, j = blocks[idx]
        flag = int(flags[idx])

        if flag == FLAG_SKIP:
            continue

        block = work[i:i + BLOCKSIZE, j:j + BLOCKSIZE]
        try:
            Qm, Rm = _qr_canonical(block)

            if flag == FLAG_Q:
                llr_raw = _llr_q_from_Q(Qm)
                llr = _norm_llr_q(llr_raw)
            elif flag == FLAG_R:
                llr_raw = _llr_r_from_R(Rm, q_eff)
                llr = _norm_llr_r(llr_raw, q_eff)
            else:
                continue

            if idx < DEBUG_BLOCK_PRINT_LIMIT:
                _debug(
                    f"[EXTRACT-INTERNAL][blk={idx}] pos=({i},{j}) flag={flag} "
                    f"llr_raw={llr_raw:.6f} llr_norm={llr:.6f}"
                )

        except Exception as e:
            if idx < DEBUG_BLOCK_PRINT_LIMIT:
                _debug(f"[EXTRACT-INTERNAL][blk={idx}] pos=({i},{j}) exception={e}")
            continue

        kpos = idx % L
        sum_score[kpos] += llr
        sum_abs[kpos] += abs(llr)
        sum_count[kpos] += 1.0

    valid = (sum_count > 0)

    out_bits = np.zeros(L, dtype=np.uint8)
    out_bits[valid] = (sum_score[valid] >= 0.0).astype(np.uint8)

    bit_conf = np.zeros(L, dtype=np.float64)
    bit_conf[valid] = np.abs(sum_score[valid]) / (sum_abs[valid] + EPS_CONF)

    conf = float(bit_conf[valid].mean()) if np.any(valid) else 0.0

    valid_bits = int(np.sum(valid))
    avg_votes = float(np.mean(sum_count[valid])) if np.any(valid) else 0.0
    min_votes = float(np.min(sum_count[valid])) if np.any(valid) else 0.0
    max_votes = float(np.max(sum_count[valid])) if np.any(valid) else 0.0

    _debug(
        f"[EXTRACT-INTERNAL] valid_bits={valid_bits}/{L}, "
        f"avg_votes={avg_votes:.4f}, min_votes={min_votes:.4f}, max_votes={max_votes:.4f}, "
        f"conf={conf:.4f}"
    )

    if np.any(valid):
        _debug(
            f"[EXTRACT-INTERNAL] bit_conf -> "
            f"mean={float(np.mean(bit_conf[valid])):.6f}, "
            f"min={float(np.min(bit_conf[valid])):.6f}, "
            f"max={float(np.max(bit_conf[valid])):.6f}"
        )

    wm_extracted = (out_bits.reshape(original_wm_shape) * 255).astype(np.uint8)
    descrambled = invert_arnold_transform(wm_extracted, arnold_iter)

    return descrambled, conf


def extract(watermarked_path, flag_path, output_path, original_wm_shape=(WM_SIZE, WM_SIZE), arnold_iter=10):
    wm_img = cv2.imread(watermarked_path)
    if wm_img is None:
        raise ValueError(f"Cannot read watermarked image: {watermarked_path}")

    flags = _load_flags(flag_path)
    if len(flags) == 0:
        raise ValueError(f"Flag file empty or missing: {flag_path}")

    wm_channel_u8 = wm_img[:, :, 0]
    H_img, W_img = wm_channel_u8.shape

    H0 = (H_img // BLOCKSIZE) * BLOCKSIZE
    W0 = (W_img // BLOCKSIZE) * BLOCKSIZE
    if H0 == 0 or W0 == 0:
        raise ValueError("Image too small for BLOCKSIZE constraint.")

    work_raw_u8 = wm_channel_u8[:H0, :W0].copy()
    total_blocks = (H0 // BLOCKSIZE) * (W0 // BLOCKSIZE)
    L = original_wm_shape[0] * original_wm_shape[1]
    odd_repeat = _highest_odd_repeat(total_blocks, L)
    selected_blocks = L * odd_repeat
    blocks = generate_block_indices(work_raw_u8.shape, BLOCKSIZE, selected_blocks, PRIVATE_KEY)

    q_eff = max(float(q), 1e-6)

    n = min(len(blocks), len(flags))
    if n != len(blocks) or n != len(flags):
        print(f"[WARN] blocks={len(blocks)} but flags={len(flags)} -> using n={n}")

    q_used, r_used, skip_used = _flag_stats(flags[:n])
    _debug(f"[EXTRACT] image='{os.path.basename(watermarked_path)}'")
    _debug(f"[EXTRACT] size={W_img}x{H_img}, working_size={W0}x{H0}")
    _debug(f"[EXTRACT] loaded_flags={len(flags)}, used_flags={n}")
    _debug(f"[EXTRACT] Q used={q_used}, R used={r_used}, SKIP used={skip_used}")
    _debug(f"[EXTRACT] total_blocks={total_blocks}, selected_blocks={len(blocks)}, payload_bits={L}")
    _debug(f"[EXTRACT] full_repeat={total_blocks // L}, odd_repeat={odd_repeat}")
    _debug(f"[EXTRACT] repeat_factor={len(blocks) / float(L):.4f}x")

    best_wm, best_conf = _extract_from_work(
        work_raw_u8.astype(np.float64),
        flags, blocks, q_eff, original_wm_shape, arnold_iter
    )
    best_name = "raw"

    if ATTACK_AWARE:
        print(f"[ATTACK-AWARE] conf(raw)        = {best_conf:.4f}")

    if ATTACK_AWARE and (NLM_ALWAYS_COMPARE or best_conf < CONF_THRESH):
        try:
            work_nlm1_u8 = _nlm_denoise_gray(
                work_raw_u8,
                h=NLM_H1,
                template_win=NLM_TEMPLATE_WIN1,
                search_win=NLM_SEARCH_WIN1
            )

            wm1, conf1 = _extract_from_work(
                work_nlm1_u8.astype(np.float64),
                flags, blocks, q_eff, original_wm_shape, arnold_iter
            )

            print(f"[ATTACK-AWARE] conf(nlm_mild)   = {conf1:.4f} (h={NLM_H1}, tw={NLM_TEMPLATE_WIN1}, sw={NLM_SEARCH_WIN1})")

            if conf1 > best_conf:
                best_wm = wm1
                best_conf = conf1
                best_name = "nlm_mild"

        except Exception as e:
            print(f"[ATTACK-AWARE] mild NLM failed: {e}")

        try:
            work_nlm2_u8 = _nlm_denoise_gray(
                work_raw_u8,
                h=NLM_H2,
                template_win=NLM_TEMPLATE_WIN2,
                search_win=NLM_SEARCH_WIN2
            )

            wm2, conf2 = _extract_from_work(
                work_nlm2_u8.astype(np.float64),
                flags, blocks, q_eff, original_wm_shape, arnold_iter
            )

            print(f"[ATTACK-AWARE] conf(nlm_strong) = {conf2:.4f} (h={NLM_H2}, tw={NLM_TEMPLATE_WIN2}, sw={NLM_SEARCH_WIN2})")

            if conf2 > best_conf:
                best_wm = wm2
                best_conf = conf2
                best_name = "nlm_strong"

        except Exception as e:
            print(f"[ATTACK-AWARE] strong NLM failed: {e}")

    if ATTACK_AWARE:
        print(f"[ATTACK-AWARE] selected='{best_name}' best_conf={best_conf:.4f}")

    _debug(f"[EXTRACT] output='{output_path}'")

    cv2.imwrite(output_path, best_wm)
    return best_wm


def calculate_metrics(original, watermarked, extracted_watermark, original_watermark_binary):
    psnr_value, ssim_value, nc_value = float("nan"), float("nan"), float("nan")

    try:
        if original is not None and watermarked is not None and original.shape == watermarked.shape:
            ori_f = original.astype(np.float64)
            wm_f = watermarked.astype(np.float64)
            mse = np.mean((ori_f - wm_f) ** 2)
            psnr_value = float("inf") if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))

            ssim_value = ssim(
                original.astype(np.uint8),
                watermarked.astype(np.uint8),
                data_range=255,
                channel_axis=-1,
            )
    except Exception as e:
        print(f"Metrics Error (PSNR/SSIM): {e}")
        traceback.print_exc()

    try:
        if original_watermark_binary is not None and extracted_watermark is not None:
            if original_watermark_binary.shape != extracted_watermark.shape:
                extracted_watermark = cv2.resize(
                    extracted_watermark,
                    (original_watermark_binary.shape[1], original_watermark_binary.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                )

            ori_wm_flat = (original_watermark_binary.flatten() > 127).astype(np.float64)
            ext_wm_flat = (extracted_watermark.flatten() > 127).astype(np.float64)

            sum_ori_sq = np.sum(ori_wm_flat ** 2)
            sum_ext_sq = np.sum(ext_wm_flat ** 2)

            if sum_ori_sq == 0 or sum_ext_sq == 0:
                nc_value = 1.0 if sum_ori_sq == sum_ext_sq else 0.0
            else:
                numerator = np.sum(ori_wm_flat * ext_wm_flat)
                denominator = np.sqrt(sum_ori_sq * sum_ext_sq)
                nc_value = numerator / denominator if denominator != 0 else 0.0

    except Exception as e:
        print(f"Metrics Error (NC): {e}")
        traceback.print_exc()

    return psnr_value, ssim_value, nc_value


def process_images(arnold_iter=10):
    input_folder = "/content"
    watermark_path = "/content/Cc.logo.circle.png"

    original_wm_raw = load_wm_gray_rgba_safe(watermark_path, WM_SIZE)
    _, original_wm_binary = cv2.threshold(original_wm_raw, 127, 255, cv2.THRESH_BINARY)

    print("Watermark path:", watermark_path)
    print("WM ones ratio:", float((original_wm_binary > 127).mean()))
    print("BLOCKSIZE:", BLOCKSIZE, "Q_TAU:", Q_TAU)
    print("Selection score: attack-aware candidate score")
    print("Mode choices: Q-candidate vs R-candidate")
    print("QR-based block embedding")
    print("Attack-aware extraction: raw vs NLM-mild vs NLM-strong")

    version_name = "qr_bs2_png64_attackaware_choice_nlm"

    out_img_dir = f"output_{version_name}_img"
    out_wm_dir = f"output_{version_name}_wm"
    out_flag_dir = f"output_{version_name}_flag"

    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_wm_dir, exist_ok=True)
    os.makedirs(out_flag_dir, exist_ok=True)

    valid_extensions = (".bmp",)
    image_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(valid_extensions) and not f.startswith("w_")
    ]

    print("Found BMP files:", image_files)
    print("Input folder:", input_folder)
    print("Output image folder:", out_img_dir)
    print("Output watermark folder:", out_wm_dir)
    print("Output flag folder:", out_flag_dir)

    for filename in image_files:
        image_path = os.path.join(input_folder, filename)
        original_img = cv2.imread(image_path)
        if original_img is None:
            print(f"Skip {filename}: read error.")
            continue

        print(f"\n--- Processing: {filename} ---")
        print(f"Input path: {image_path}")
        print(f"Input shape: {original_img.shape}")

        base = os.path.splitext(filename)[0]
        watermarked_path = os.path.join(out_img_dir, f"watermarked_{base}.png")
        flag_path = os.path.join(out_flag_dir, f"flag_{base}.txt")
        extracted_wm_path = os.path.join(out_wm_dir, f"extracted_{base}.png")

        print(f"Watermarked output : {watermarked_path}")
        print(f"Flag output        : {flag_path}")
        print(f"Extracted WM output: {extracted_wm_path}")

        watermarked_img, extracted_wm = None, None

        print("Embedding...")
        t0 = time.time()
        try:
            watermarked_img = embed(image_path, watermark_path, watermarked_path, flag_path, arnold_iter)
        except Exception as e:
            print(f"Embedding failed for {filename}: {e}")
            traceback.print_exc()
        embed_time = time.time() - t0

        if os.path.exists(flag_path):
            dbg_flags = _load_flags(flag_path)
            q_used, r_used, skip_used = _flag_stats(dbg_flags)
            print(f"Flag summary for {filename}: Q={q_used}, R={r_used}, SKIP={skip_used}, TOTAL={len(dbg_flags)}")

        print("Extracting...")
        t0 = time.time()
        try:
            if watermarked_img is not None and os.path.exists(flag_path):
                extracted_wm = extract(watermarked_path, flag_path, extracted_wm_path, (WM_SIZE, WM_SIZE), arnold_iter)
        except Exception as e:
            print(f"Extraction failed for {filename}: {e}")
            traceback.print_exc()
        extract_time = time.time() - t0

        print("Calculating metrics...")
        psnr_val, ssim_val, nc_val = calculate_metrics(original_img, watermarked_img, extracted_wm, original_wm_binary)

        print(
            f"Metrics for {filename}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, NC: {nc_val:.4f}, "
            f"Embed: {embed_time:.4f}s, Extract: {extract_time:.4f}s"
        )


if __name__ == "__main__":
    process_images(ARNOLD_ITER)
    print("\nAll processing finished.")

Watermark path: /content/Cc.logo.circle.png
WM ones ratio: 0.656494140625
BLOCKSIZE: 2 Q_TAU: 0.5
Selection score: attack-aware candidate score
Mode choices: Q-candidate vs R-candidate
QR-based block embedding
Attack-aware extraction: raw vs NLM-mild vs NLM-strong
Found BMP files: ['Girl.bmp', 'lenna.bmp', 'manhatan.bmp', 'airplane.bmp', 'pepper.bmp', 'safari.bmp']
Input folder: /content
Output image folder: output_qr_bs2_png64_attackaware_choice_nlm_img
Output watermark folder: output_qr_bs2_png64_attackaware_choice_nlm_wm
Output flag folder: output_qr_bs2_png64_attackaware_choice_nlm_flag

--- Processing: Girl.bmp ---
Input path: /content/Girl.bmp
Input shape: (512, 512, 3)
Watermarked output : output_qr_bs2_png64_attackaware_choice_nlm_img/watermarked_Girl.png
Flag output        : output_qr_bs2_png64_attackaware_choice_nlm_flag/flag_Girl.txt
Extracted WM output: output_qr_bs2_png64_attackaware_choice_nlm_wm/extracted_Girl.png
Embedding...
[EMBED] host='Girl.bmp'
[EMBED] original_siz

In [5]:
# -*- coding: utf-8 -*-

import os, time, math, json, csv, traceback
from typing import Tuple, Dict, Any, Optional, List, Union

import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim


WM_SIZE = globals().get("WM_SIZE", 64)
ARNOLD_ITER = globals().get("ARNOLD_ITER", 10)


def prepare_watermark_for_opencv(
    watermark_path_in: str,
    out_binary_png: str,
    wm_size: int,
    svg_raster_size: int = 512,
    bg: int = 255
) -> str:
    """
    Converts watermark input (supports .svg, RGBA PNG) into a clean binary PNG (0/255),
    grayscale, size (wm_size, wm_size), with alpha composited onto white background.

    If input is .svg, requires: pip install cairosvg
    """
    path = watermark_path_in
    ext = os.path.splitext(path)[1].lower()

    if ext == ".svg":
        try:
            import cairosvg
        except Exception as e:
            raise RuntimeError(
                "SVG watermark needs cairosvg. Install once: pip install cairosvg"
            ) from e

        tmp_png = os.path.splitext(out_binary_png)[0] + "_raster.png"
        cairosvg.svg2png(
            url=path,
            write_to=tmp_png,
            output_width=int(svg_raster_size),
            output_height=int(svg_raster_size),
        )
        path = tmp_png

    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot read watermark: {watermark_path_in} (after raster: {path})")

    if img.ndim == 3 and img.shape[2] == 4:
        bgr = img[:, :, :3].astype(np.float32)
        a = img[:, :, 3:4].astype(np.float32) / 255.0
        bg_bgr = np.full_like(bgr, float(bg), dtype=np.float32)
        comp = bgr * a + bg_bgr * (1.0 - a)
        gray = cv2.cvtColor(comp.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    elif img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    gray = cv2.resize(gray, (wm_size, wm_size), interpolation=cv2.INTER_AREA)
    _, bin_wm = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

    out_dir = os.path.dirname(out_binary_png)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    cv2.imwrite(out_binary_png, bin_wm)
    return out_binary_png


def _binarize01(img: np.ndarray) -> np.ndarray:
    if img is None:
        return None
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return (img.astype(np.uint8) >= 128).astype(np.uint8)


def align_extracted_to_gt_binary(extracted_wm: np.ndarray, wm_gt_bin: np.ndarray) -> np.ndarray:
    if extracted_wm is None or wm_gt_bin is None:
        return extracted_wm

    gt01 = _binarize01(wm_gt_bin)
    ex01 = _binarize01(extracted_wm)

    if ex01.shape != gt01.shape:
        ex01 = cv2.resize(ex01, (gt01.shape[1], gt01.shape[0]), interpolation=cv2.INTER_NEAREST)

    same = float(np.mean(ex01 == gt01))
    inv = float(np.mean((1 - ex01) == gt01))

    if inv > same:
        ex01 = 1 - ex01

    return (ex01 * 255).astype(np.uint8)


def psnr_uint8(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    mse = np.mean((a - b) ** 2)
    if mse <= 1e-12:
        return 99.0
    return 10.0 * math.log10((255.0 ** 2) / mse)


def ssim_uint8(a: np.ndarray, b: np.ndarray) -> float:
    if a.ndim == 2:
        return float(ssim(a, b, data_range=255))
    try:
        return float(ssim(a, b, data_range=255, channel_axis=-1))
    except TypeError:
        return float(ssim(a, b, data_range=255, multichannel=True))


def nc_binary(wm_gt: np.ndarray, wm_ex: np.ndarray) -> float:
    if wm_gt is None or wm_ex is None:
        return float("nan")

    gt01 = _binarize01(wm_gt).astype(np.float64).ravel()
    ex01 = _binarize01(wm_ex).astype(np.float64)

    if wm_ex.ndim == 3:
        wm_ex_gray = cv2.cvtColor(wm_ex, cv2.COLOR_BGR2GRAY)
    else:
        wm_ex_gray = wm_ex

    if wm_gt.ndim == 3:
        wm_gt_gray = cv2.cvtColor(wm_gt, cv2.COLOR_BGR2GRAY)
    else:
        wm_gt_gray = wm_gt

    if wm_ex_gray.shape != wm_gt_gray.shape:
        ex01 = cv2.resize(
            ex01.astype(np.uint8),
            (wm_gt_gray.shape[1], wm_gt_gray.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.float64)
    ex01 = ex01.ravel()

    num = float(np.sum(gt01 * ex01))
    den = float(np.sqrt(np.sum(gt01 * gt01) * np.sum(ex01 * ex01)))
    nc1 = (num / den) if den > 1e-12 else (1.0 if np.array_equal(gt01, ex01) else 0.0)

    exi = 1.0 - ex01
    num2 = float(np.sum(gt01 * exi))
    den2 = float(np.sqrt(np.sum(gt01 * gt01) * np.sum(exi * exi)))
    nc2 = (num2 / den2) if den2 > 1e-12 else (1.0 if np.array_equal(gt01, exi) else 0.0)

    return max(nc1, nc2)


def calculate_metrics(host_img: np.ndarray, attacked_img: np.ndarray, extracted_wm: np.ndarray, wm_gt_bin: np.ndarray):
    p = psnr_uint8(host_img, attacked_img)
    s = ssim_uint8(host_img, attacked_img)
    n = nc_binary(wm_gt_bin, extracted_wm)
    return p, s, n


def _attack_dir_name(attack_type: str) -> str:
    return (
        attack_type.lower()
        .replace(" ", "_")
        .replace("&", "and")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "_")
        .replace(".", "p")
    )


def ensure_uint8(img):
    if img is None:
        return None
    if img.dtype == np.uint8:
        return img
    img = np.clip(img, 0, 255)
    return img.astype(np.uint8)


def _fmt_num(x) -> str:
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        s = f"{float(x):.6f}".rstrip("0").rstrip(".")
        if s == "-0":
            s = "0"
        return s.replace(".", "p").replace("-", "m")
    return str(x)


def _fmt_params(p) -> str:
    if p is None:
        return "None"
    if isinstance(p, tuple):
        return "(" + ",".join(_fmt_params(x) for x in p) + ")"
    if isinstance(p, list):
        return "[" + ",".join(_fmt_params(x) for x in p) + "]"
    if isinstance(p, float):
        return f"{p:.6f}".rstrip("0").rstrip(".")
    return str(p)


def _metric_clean(x):
    if x is None:
        return None
    try:
        v = float(x)
    except:
        return None
    if math.isnan(v):
        return None
    if math.isinf(v):
        return 99.0
    return v


def _safe_mean(vals: List[Any]) -> Optional[float]:
    buf = []
    for v in vals:
        vv = _metric_clean(v)
        if vv is None:
            continue
        buf.append(vv)
    if not buf:
        return None
    return float(sum(buf) / len(buf))


def scale_attack(image: np.ndarray, scale_factor: float) -> np.ndarray:
    if scale_factor <= 0:
        raise ValueError("scale_factor must be > 0")
    h, w = image.shape[:2]
    new_w = max(1, int(round(w * scale_factor)))
    new_h = max(1, int(round(h * scale_factor)))
    interp1 = cv2.INTER_AREA if scale_factor < 1.0 else cv2.INTER_CUBIC
    tmp = cv2.resize(image, (new_w, new_h), interpolation=interp1)
    interp2 = cv2.INTER_CUBIC if scale_factor < 1.0 else cv2.INTER_AREA
    out = cv2.resize(tmp, (w, h), interpolation=interp2)
    return out.astype(np.uint8)


def salt_pepper_attack(image: np.ndarray, noise_density: float, s_vs_p: float = 0.5, seed: int = 0) -> np.ndarray:
    if not (0.0 <= noise_density <= 1.0):
        raise ValueError("noise_density must be in [0,1]")
    if not (0.0 <= s_vs_p <= 1.0):
        raise ValueError("s_vs_p must be in [0,1]")
    rng = np.random.default_rng(seed)
    out = image.copy()
    h, w = out.shape[:2]
    num = int(round(noise_density * h * w))
    num_salt = int(round(num * s_vs_p))
    num_pepper = num - num_salt
    ys = rng.integers(0, h, size=num_salt, endpoint=False)
    xs = rng.integers(0, w, size=num_salt, endpoint=False)
    yp = rng.integers(0, h, size=num_pepper, endpoint=False)
    xp = rng.integers(0, w, size=num_pepper, endpoint=False)
    if out.ndim == 2:
        out[ys, xs] = 255
        out[yp, xp] = 0
    else:
        out[ys, xs, :] = 255
        out[yp, xp, :] = 0
    return out.astype(np.uint8)


def gaussian_noise_attack(image: np.ndarray, mean: float, variance: float, domain: str = "norm", seed: int = 0) -> np.ndarray:
    if variance < 0:
        raise ValueError("variance must be >= 0")
    rng = np.random.default_rng(seed)
    sigma = math.sqrt(variance)
    if domain == "pixel":
        img_f = image.astype(np.float32)
        noise = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out = np.clip(img_f + noise, 0.0, 255.0).astype(np.uint8)
        return out
    if domain == "norm":
        img_n = image.astype(np.float32) / 255.0
        noise = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out_n = np.clip(img_n + noise, 0.0, 1.0)
        out = (out_n * 255.0 + 0.5).astype(np.uint8)
        return out
    raise ValueError("domain must be 'norm' or 'pixel'")


def speckle_noise_attack(image: np.ndarray, mean: float, variance: float, domain: str = "norm", seed: int = 0) -> np.ndarray:
    if variance < 0:
        raise ValueError("variance must be >= 0")
    rng = np.random.default_rng(seed)
    sigma = math.sqrt(variance)
    if domain == "pixel":
        img_f = image.astype(np.float32)
        n = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out = np.clip(img_f + img_f * n, 0.0, 255.0)
        return out.astype(np.uint8)
    if domain == "norm":
        img_n = image.astype(np.float32) / 255.0
        n = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out_n = np.clip(img_n + img_n * n, 0.0, 1.0)
        out = (out_n * 255.0 + 0.5).astype(np.uint8)
        return out
    raise ValueError("domain must be 'norm' or 'pixel'")


def blur_attack(image: np.ndarray, sigma: float) -> np.ndarray:
    if sigma < 0:
        raise ValueError("sigma must be >= 0")
    if sigma == 0:
        return image.copy()
    return cv2.GaussianBlur(image, (0, 0), sigmaX=sigma, sigmaY=sigma).astype(np.uint8)


def average_filter_attack(image: np.ndarray, ksize: int) -> np.ndarray:
    if ksize < 1:
        raise ValueError("ksize must be >= 1")
    return cv2.blur(image, (ksize, ksize)).astype(np.uint8)


def median_filter_attack(image: np.ndarray, ksize: int) -> np.ndarray:
    if ksize < 3 or (ksize % 2 == 0):
        raise ValueError("median ksize must be odd and >= 3 (e.g., 3,5,7)")
    return cv2.medianBlur(image, ksize).astype(np.uint8)


def histogram_attack(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return cv2.equalizeHist(image).astype(np.uint8)
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
    out = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return out.astype(np.uint8)


def sharpen_attack(image: np.ndarray, sigma: float, amount: float = 1.5) -> np.ndarray:
    if sigma <= 0:
        return image.copy()
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=sigma, sigmaY=sigma)
    out = cv2.addWeighted(image, 1.0 + amount, blurred, -amount, 0.0)
    return np.clip(out, 0, 255).astype(np.uint8)


def rotation_attack(
    image: np.ndarray,
    angle_deg: float,
    inverse_back: bool = True,
    border_mode: int = cv2.BORDER_CONSTANT,
    border_value: int = 0
) -> np.ndarray:
    h, w = image.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(
        image, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=border_mode,
        borderValue=border_value
    )
    if inverse_back:
        Minv = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)
        rotated = cv2.warpAffine(
            rotated, Minv, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=border_mode,
            borderValue=border_value
        )
    return rotated.astype(np.uint8)


def translation_attack(
    image: np.ndarray,
    shift_x: int,
    shift_y: int,
    inverse_back: bool = False,
    border_mode: int = cv2.BORDER_CONSTANT,
    border_value: int = 0
) -> np.ndarray:
    h, w = image.shape[:2]
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    shifted = cv2.warpAffine(
        image, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=border_mode,
        borderValue=border_value
    )
    if inverse_back:
        Minv = np.float32([[1, 0, -shift_x], [0, 1, -shift_y]])
        shifted = cv2.warpAffine(
            shifted, Minv, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=border_mode,
            borderValue=border_value
        )
    return shifted.astype(np.uint8)


def jpeg_attack(image: np.ndarray, quality_factor: int) -> Tuple[np.ndarray, int]:
    q = int(np.clip(quality_factor, 0, 100))
    params = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    ok, enc = cv2.imencode(".jpg", image, params)
    if not ok:
        raise RuntimeError("cv2.imencode JPEG failed")
    flag = cv2.IMREAD_COLOR if image.ndim == 3 else cv2.IMREAD_GRAYSCALE
    out = cv2.imdecode(enc, flag)
    if out is None:
        raise RuntimeError("cv2.imdecode JPEG failed")
    return out.astype(np.uint8), int(enc.size)


def jp2_attack_ratio(
    image: np.ndarray,
    compression_ratio: float,
    force_lossy: bool = True,
    max_tries: int = 10
) -> Tuple[np.ndarray, Dict[str, Any]]:
    dbg: Dict[str, Any] = {}
    ratio_req = float(compression_ratio)
    if ratio_req < 1.0:
        ratio_req = 1.0
    dbg["compression_ratio_req"] = ratio_req
    dbg["raw_bytes"] = int(image.size)
    dbg["method"] = "opencv"
    dbg["force_lossy"] = bool(force_lossy)

    def enc_dec(rate_x1000: int) -> Tuple[np.ndarray, int]:
        params = [int(cv2.IMWRITE_JPEG2000_COMPRESSION_X1000), int(rate_x1000)]
        ok, enc = cv2.imencode(".jp2", image, params)
        if not ok:
            raise RuntimeError("JPEG2000 encode failed (OpenCV build may lack JP2 support).")
        flag = cv2.IMREAD_COLOR if image.ndim == 3 else cv2.IMREAD_GRAYSCALE
        out = cv2.imdecode(enc, flag)
        if out is None:
            raise RuntimeError("JPEG2000 decode failed.")
        return out.astype(np.uint8), int(enc.size)

    out_lossless, bytes_lossless = enc_dec(1000)
    dbg["lossless_test_rate_x1000"] = 1000
    dbg["lossless_jp2_bytes"] = bytes_lossless
    dbg["lossless_decoded_identical"] = bool(np.array_equal(out_lossless, image))

    rate = int(round(1000.0 / ratio_req))
    rate = max(1, min(rate, 1000))
    dbg["rate_x1000_initial"] = rate

    out, jp2_bytes = enc_dec(rate)
    dbg["jp2_bytes_initial"] = jp2_bytes
    dbg["jp2_identical_initial"] = bool(np.array_equal(out, image))
    dbg["psnr_initial(dB)"] = psnr_uint8(image, out)

    if not force_lossy:
        dbg["rate_x1000_final"] = rate
        dbg["jp2_bytes_final"] = jp2_bytes
        dbg["jp2_identical_final"] = bool(np.array_equal(out, image))
        dbg["psnr_final(dB)"] = psnr_uint8(image, out)
        return out, dbg

    tries = 0
    rate_cur = rate
    out_cur = out
    bytes_cur = jp2_bytes

    while tries < max_tries and np.array_equal(out_cur, image) and rate_cur > 1:
        tries += 1
        rate_cur = max(1, int(round(rate_cur * 0.75)))
        out_cur, bytes_cur = enc_dec(rate_cur)

    dbg["tries"] = tries
    dbg["rate_x1000_final"] = rate_cur
    dbg["jp2_bytes_final"] = bytes_cur
    dbg["jp2_identical_final"] = bool(np.array_equal(out_cur, image))
    dbg["psnr_final(dB)"] = psnr_uint8(image, out_cur)
    return out_cur, dbg


def crop_resize_attack_area(image: np.ndarray, crop_area_percent: float, where: str = "center") -> np.ndarray:
    if not (0.0 <= crop_area_percent < 100.0):
        raise ValueError("crop_area_percent must be in [0,100)")
    h, w = image.shape[:2]
    p = crop_area_percent / 100.0
    if p == 0.0:
        return image.copy()
    keep_area = 1.0 - p
    keep_side = math.sqrt(max(0.0, keep_area))
    rh = max(1, min(h, int(round(h * keep_side))))
    rw = max(1, min(w, int(round(w * keep_side))))

    def origin(H: int, W: int, rh_: int, rw_: int, where_: str) -> Tuple[int, int]:
        where_ = where_.lower()
        if where_ == "center":
            return (H - rh_) // 2, (W - rw_) // 2
        if where_ == "top":
            return 0, (W - rw_) // 2
        if where_ == "bottom":
            return H - rh_, (W - rw_) // 2
        if where_ == "left":
            return (H - rh_) // 2, 0
        if where_ == "right":
            return (H - rh_) // 2, W - rw_
        if where_ == "top-left":
            return 0, 0
        if where_ == "top-right":
            return 0, W - rw_
        if where_ == "bottom-left":
            return H - rh_, 0
        if where_ == "bottom-right":
            return H - rh_, W - rw_
        raise ValueError("unknown where")

    y0, x0 = origin(h, w, rh, rw, where)
    cropped = image[y0:y0 + rh, x0:x0 + rw]
    out = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)
    return out.astype(np.uint8)


OcclValue = Union[int, Tuple[int, int, int], str]


def occlusion_attack_area(
    image: np.ndarray,
    occlude_area_percent: float,
    where: str = "top-left",
    value: OcclValue = 0,
    seed: int = 0
) -> np.ndarray:
    if not (0.0 <= occlude_area_percent < 100.0):
        raise ValueError("occlude_area_percent must be in [0,100)")
    h, w = image.shape[:2]
    p = occlude_area_percent / 100.0
    if p == 0.0:
        return image.copy()

    side = math.sqrt(p)
    rh = max(1, min(h, int(round(h * side))))
    rw = max(1, min(w, int(round(w * side))))
    rng = np.random.default_rng(int(seed))

    def origin(H: int, W: int, rh_: int, rw_: int, where_: str) -> Tuple[int, int]:
        where_ = where_.lower()
        if where_ in {"random", "rand"}:
            y0 = int(rng.integers(0, max(1, H - rh_ + 1)))
            x0 = int(rng.integers(0, max(1, W - rw_ + 1)))
            return y0, x0
        if where_ == "center":
            return (H - rh_) // 2, (W - rw_) // 2
        if where_ == "top-left":
            return 0, 0
        if where_ == "top-right":
            return 0, W - rw_
        if where_ == "bottom-left":
            return H - rh_, 0
        if where_ == "bottom-right":
            return H - rh_, W - rw_
        if where_ == "top":
            return 0, (W - rw_) // 2
        if where_ == "bottom":
            return H - rh_, (W - rw_) // 2
        if where_ == "left":
            return (H - rh_) // 2, 0
        if where_ == "right":
            return (H - rh_) // 2, W - rw_
        raise ValueError("unknown where")

    out = image.copy()
    y0, x0 = origin(h, w, rh, rw, where)

    fill = value
    if isinstance(value, str):
        v = value.lower()
        if v == "mean":
            if out.ndim == 2:
                fill = int(np.round(float(np.mean(out))))
            else:
                m = np.mean(out.reshape(-1, 3), axis=0)
                fill = (int(round(m[0])), int(round(m[1])), int(round(m[2])))
        elif v == "random":
            if out.ndim == 2:
                patch = rng.integers(0, 256, size=(rh, rw), dtype=np.uint8)
                out[y0:y0 + rh, x0:x0 + rw] = patch
                return out.astype(np.uint8)
            else:
                patch = rng.integers(0, 256, size=(rh, rw, 3), dtype=np.uint8)
                out[y0:y0 + rh, x0:x0 + rw, :] = patch
                return out.astype(np.uint8)
        else:
            raise ValueError("value string must be 'mean' or 'random'")

    if out.ndim == 2:
        out[y0:y0 + rh, x0:x0 + rw] = int(fill)
    else:
        if isinstance(fill, (tuple, list)):
            if len(fill) != 3:
                raise ValueError("color fill tuple must be (B,G,R)")
            out[y0:y0 + rh, x0:x0 + rw, :] = np.array(fill, dtype=np.uint8)
        else:
            out[y0:y0 + rh, x0:x0 + rw, :] = int(fill)

    return out.astype(np.uint8)


def lowpass_filter_attack(image: np.ndarray, kernel_size_x: int, kernel_size_y: int) -> np.ndarray:
    if kernel_size_x <= 0 or kernel_size_y <= 0:
        raise ValueError("kernel sizes must be positive")
    if kernel_size_x % 2 == 0:
        kernel_size_x += 1
    if kernel_size_y % 2 == 0:
        kernel_size_y += 1
    return cv2.GaussianBlur(image, (kernel_size_x, kernel_size_y), 0).astype(np.uint8)


def gamma_attack(image: np.ndarray, gamma: float) -> np.ndarray:
    if gamma <= 0:
        raise ValueError("gamma must be > 0")
    inv = 1.0 / float(gamma)
    lut = np.array([((i / 255.0) ** inv) * 255.0 for i in range(256)], dtype=np.float32)
    lut = np.clip(lut + 0.5, 0, 255).astype(np.uint8)
    return cv2.LUT(image, lut).astype(np.uint8)


def motion_blur_attack(image: np.ndarray, ksize: int, direction: str = "h") -> np.ndarray:
    if ksize < 1:
        raise ValueError("ksize must be >= 1")
    ksize = int(ksize)
    if ksize % 2 == 0:
        ksize += 1
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    direction = direction.lower()
    if direction == "h":
        kernel[ksize // 2, :] = 1.0
    elif direction == "v":
        kernel[:, ksize // 2] = 1.0
    else:
        raise ValueError("direction must be 'h' or 'v'")
    kernel /= kernel.sum()
    out = cv2.filter2D(image, -1, kernel)
    return np.clip(out, 0, 255).astype(np.uint8)


def brightness_contrast_attack(image: np.ndarray, alpha: float = 1.0, beta: float = 0.0) -> np.ndarray:
    img_f = image.astype(np.float32)
    out = img_f * float(alpha) + float(beta)
    return np.clip(out + 0.5, 0, 255).astype(np.uint8)


def clahe_y_attack(image: np.ndarray, clip_limit: float = 2.0, tile_grid: int = 8) -> np.ndarray:
    if image.ndim == 2:
        clahe = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=(int(tile_grid), int(tile_grid)))
        return clahe.apply(image).astype(np.uint8)
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    clahe = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=(int(tile_grid), int(tile_grid)))
    ycrcb[:, :, 0] = clahe.apply(ycrcb[:, :, 0])
    out = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return out.astype(np.uint8)


def bilateral_attack(image: np.ndarray, d: int = 7, sigma_color: float = 30.0, sigma_space: float = 7.0) -> np.ndarray:
    out = cv2.bilateralFilter(image, int(d), float(sigma_color), float(sigma_space))
    return np.clip(out, 0, 255).astype(np.uint8)


ATTACK_SUITE = [
    ("Blur", 1),
    ("Sharpen", (1, 1.5)),
    ("Gaussian Noise", (0.0, 0.003)),
    ("Salt & Pepper", 0.1),
    ("JPEG", 50),
    ("JPEG2000", 13),
    ("Lowpass", (9, 9)),
    ("Scale", 0.2),
    ("Rotation", 45),
    ("Gamma", 1.5),
    ("Occlusion", (50.0, "random", 0, 3)),
]


def apply_attack(wm_img: np.ndarray, attack_type: str, attack_param):
    extra_dbg = None

    if attack_type == "Blur":
        sigma = float(attack_param)
        out = blur_attack(wm_img, sigma)
        pstr = f"_sigma{_fmt_num(sigma)}"
        return out, pstr, extra_dbg

    if attack_type == "Sharpen":
        sigma, amount = attack_param if isinstance(attack_param, tuple) else (float(attack_param), 1.5)
        sigma = float(sigma)
        amount = float(amount)
        out = sharpen_attack(wm_img, sigma, amount=amount)
        pstr = f"_sigma{_fmt_num(sigma)}_a{_fmt_num(amount)}"
        return out, pstr, extra_dbg

    if attack_type == "Gaussian Noise":
        mean, var = attack_param
        out = gaussian_noise_attack(wm_img, float(mean), float(var), domain="norm")
        pstr = f"_m{_fmt_num(mean)}_v{_fmt_num(var)}"
        return out, pstr, extra_dbg

    if attack_type == "Speckle Noise":
        mean, var = attack_param
        out = speckle_noise_attack(wm_img, float(mean), float(var), domain="norm")
        pstr = f"_m{_fmt_num(mean)}_v{_fmt_num(var)}"
        return out, pstr, extra_dbg

    if attack_type == "Salt & Pepper":
        d = float(attack_param)
        out = salt_pepper_attack(wm_img, d)
        pstr = f"_d{_fmt_num(d)}"
        return out, pstr, extra_dbg

    if attack_type == "JPEG":
        qf = int(attack_param)
        out, jpeg_bytes = jpeg_attack(wm_img, qf)
        extra_dbg = {"jpeg_bytes": int(jpeg_bytes)}
        pstr = f"_qf{qf}"
        return out, pstr, extra_dbg

    if attack_type == "JPEG2000":
        ratio = float(attack_param)
        out, dbg = jp2_attack_ratio(wm_img, ratio, force_lossy=True)
        extra_dbg = dbg
        rate_final = dbg.get("rate_x1000_final", None)
        if rate_final is not None:
            pstr = f"_cr{_fmt_num(ratio)}_r{int(rate_final)}"
        else:
            pstr = f"_cr{_fmt_num(ratio)}"
        return out, pstr, extra_dbg

    if attack_type == "Median":
        k = int(attack_param)
        out = median_filter_attack(wm_img, k)
        pstr = f"_k{k}"
        return out, pstr, extra_dbg

    if attack_type == "Average":
        k = int(attack_param)
        out = average_filter_attack(wm_img, k)
        pstr = f"_k{k}"
        return out, pstr, extra_dbg

    if attack_type == "Lowpass":
        kx, ky = attack_param
        out = lowpass_filter_attack(wm_img, int(kx), int(ky))
        pstr = f"_kx{int(kx)}_ky{int(ky)}"
        return out, pstr, extra_dbg

    if attack_type == "Scale":
        s_ = float(attack_param)
        out = scale_attack(wm_img, s_)
        pstr = f"_s{_fmt_num(s_)}"
        return out, pstr, extra_dbg

    if attack_type == "Crop":
        crop_percent, where = attack_param
        crop_percent = float(crop_percent)
        where = str(where).lower()
        out = crop_resize_attack_area(wm_img, crop_percent, where=where)
        pstr = f"_p{_fmt_num(crop_percent)}_{where}"
        return out, pstr, extra_dbg

    if attack_type == "Rotation":
        ang = float(attack_param)
        out = rotation_attack(wm_img, ang, inverse_back=True)
        pstr = f"_a{_fmt_num(ang)}"
        return out, pstr, extra_dbg

    if attack_type == "Translation":
        sx, sy = attack_param
        out = translation_attack(wm_img, int(sx), int(sy), inverse_back=False)
        pstr = f"_dx{int(sx)}_dy{int(sy)}"
        return out, pstr, extra_dbg

    if attack_type == "Gamma":
        g = float(attack_param)
        out = gamma_attack(wm_img, g)
        pstr = f"_g{_fmt_num(g)}"
        return out, pstr, extra_dbg

    if attack_type == "Motion Blur":
        ksize, direction = attack_param
        out = motion_blur_attack(wm_img, int(ksize), str(direction).lower())
        pstr = f"_k{int(ksize)}_{str(direction).lower()}"
        return out, pstr, extra_dbg

    if attack_type == "BrightnessContrast":
        alpha, beta = attack_param
        out = brightness_contrast_attack(wm_img, float(alpha), float(beta))
        pstr = f"_a{_fmt_num(alpha)}_b{_fmt_num(beta)}"
        return out, pstr, extra_dbg

    if attack_type == "CLAHE_Y":
        clip, tile = attack_param
        out = clahe_y_attack(wm_img, float(clip), int(tile))
        pstr = f"_c{_fmt_num(clip)}_t{int(tile)}"
        return out, pstr, extra_dbg

    if attack_type == "Bilateral":
        d, sc, ss_ = attack_param
        out = bilateral_attack(wm_img, int(d), float(sc), float(ss_))
        pstr = f"_d{int(d)}_sc{_fmt_num(sc)}_ss{_fmt_num(ss_)}"
        return out, pstr, extra_dbg

    if attack_type == "Histogram":
        out = histogram_attack(wm_img)
        pstr = "_eq"
        return out, pstr, extra_dbg

    if attack_type == "Occlusion":
        if isinstance(attack_param, (tuple, list)):
            if len(attack_param) == 3:
                p, where, value = attack_param
                seed = 0
            elif len(attack_param) == 4:
                p, where, value, seed = attack_param
            else:
                raise ValueError("Occlusion params must be (p, where, value) or (p, where, value, seed)")
        else:
            raise ValueError("Occlusion params must be a tuple/list")

        out = occlusion_attack_area(
            wm_img,
            float(p),
            where=str(where).lower(),
            value=value,
            seed=int(seed)
        )
        pstr = f"_p{_fmt_num(p)}_{str(where).lower()}_v{str(value).lower() if isinstance(value, str) else value}_s{int(seed)}"
        return out, pstr, extra_dbg

    raise ValueError(f"Unknown attack_type: {attack_type}")


def run_attack_suite(input_folder: str, watermark_path: str, base_dir: str = "su_new_attacks", arnold_iter: int = ARNOLD_ITER):
    os.makedirs(base_dir, exist_ok=True)

    prepared_wm_path = os.path.join(base_dir, "watermark_prepared_binary.png")
    watermark_path_use = prepare_watermark_for_opencv(
        watermark_path_in=watermark_path,
        out_binary_png=prepared_wm_path,
        wm_size=WM_SIZE,
        svg_raster_size=512,
        bg=255
    )

    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith(".bmp")])
    if not images:
        raise FileNotFoundError(f"No .bmp images found in {input_folder}")

    wm_raw = cv2.imread(watermark_path_use, cv2.IMREAD_GRAYSCALE)
    if wm_raw is None:
        raise FileNotFoundError(f"Cannot read prepared watermark: {watermark_path_use}")
    wm_raw = cv2.resize(wm_raw, (WM_SIZE, WM_SIZE), interpolation=cv2.INTER_NEAREST)
    _, wm_gt_bin = cv2.threshold(wm_raw, 127, 255, cv2.THRESH_BINARY)

    watermarked_dir = os.path.join(base_dir, "watermarked_once")
    attacked_root = os.path.join(base_dir, "attacked")
    extracted_root = os.path.join(base_dir, "extracted")
    os.makedirs(watermarked_dir, exist_ok=True)
    os.makedirs(attacked_root, exist_ok=True)
    os.makedirs(extracted_root, exist_ok=True)

    total_jobs = len(images) * len(ATTACK_SUITE)
    job_idx = 0

    print("=" * 70)
    print("AUTO RUN: Hess_new attack suite for ALL BMP images (flag_path)")
    print(f"Input folder   : {input_folder}")
    print(f"Watermark path : {watermark_path}  (prepared -> {watermark_path_use})")
    print(f"BMP images     : {len(images)}")
    print(f"Total runs     : {total_jobs}")
    print("=" * 70)

    all_results = []
    baseline_by_image = []

    for img_i, image_name in enumerate(images, 1):
        host_path = os.path.join(input_folder, image_name)
        host_img = cv2.imread(host_path, cv2.IMREAD_COLOR)
        if host_img is None:
            print(f"[SKIP] cannot read: {image_name}")
            continue

        base_name = os.path.splitext(image_name)[0]

        wm_img_path = os.path.join(watermarked_dir, f"watermarked_{base_name}.png")
        flag_path = os.path.join(watermarked_dir, f"flag_{base_name}.txt")
        wm_extract0 = os.path.join(watermarked_dir, f"extracted_baseline_{base_name}.png")

        try:
            t0 = time.time()
            wm_img = embed(host_path, watermark_path_use, wm_img_path, flag_path, arnold_iter=arnold_iter)
            wm_img = ensure_uint8(wm_img)
            t_embed = time.time() - t0

            t1 = time.time()
            ext0 = extract(wm_img_path, flag_path, wm_extract0, (WM_SIZE, WM_SIZE), arnold_iter=arnold_iter)
            ext0 = ensure_uint8(ext0)
            ext0 = align_extracted_to_gt_binary(ext0, wm_gt_bin)
            cv2.imwrite(wm_extract0, ext0)
            t_extract0 = time.time() - t1

            psnr0, ssim0, nc0 = calculate_metrics(host_img, wm_img, ext0, wm_gt_bin)
            baseline_by_image.append({
                "image": image_name,
                "psnr_orig": psnr0,
                "ssim_orig": ssim0,
                "nc_orig": nc0,
                "embed_time": t_embed,
                "extract_time_baseline": t_extract0
            })

            print(
                f"\n[{img_i}/{len(images)}] BASELINE {image_name}: "
                f"PSNR={_metric_clean(psnr0)} SSIM={_metric_clean(ssim0)} NC={_metric_clean(nc0)} "
                f"(embed {t_embed:.2f}s, extract {t_extract0:.2f}s)"
            )

        except Exception as e:
            print(f"[FAIL] baseline embed/extract for {image_name}: {e}")
            traceback.print_exc()
            continue

        for attack_type, attack_params in ATTACK_SUITE:
            job_idx += 1
            print(f"  [{job_idx}/{total_jobs}] {image_name} | {attack_type} | {attack_params}")

            atk_dir = os.path.join(attacked_root, _attack_dir_name(attack_type))
            ext_dir = os.path.join(extracted_root, _attack_dir_name(attack_type))
            os.makedirs(atk_dir, exist_ok=True)
            os.makedirs(ext_dir, exist_ok=True)

            try:
                tA = time.time()
                attacked_img, param_str, extra_dbg = apply_attack(wm_img, attack_type, attack_params)
                attacked_img = ensure_uint8(attacked_img)

                attacked_path = os.path.join(atk_dir, f"{_attack_dir_name(attack_type)}_{base_name}{param_str}.bmp")
                cv2.imwrite(attacked_path, attacked_img)

                extracted_path = os.path.join(ext_dir, f"extracted_{base_name}{param_str}.png")
                ext_wm = extract(attacked_path, flag_path, extracted_path, (WM_SIZE, WM_SIZE), arnold_iter=arnold_iter)
                ext_wm = ensure_uint8(ext_wm)
                ext_wm = align_extracted_to_gt_binary(ext_wm, wm_gt_bin)
                cv2.imwrite(extracted_path, ext_wm)

                psnr1, ssim1, nc1 = calculate_metrics(host_img, attacked_img, ext_wm, wm_gt_bin)
                dt = time.time() - tA

                row = {
                    "image_name": image_name,
                    "attack_type": attack_type,
                    "attack_params": _fmt_params(attack_params),

                    "psnr_orig": _metric_clean(psnr0),
                    "ssim_orig": _metric_clean(ssim0),
                    "nc_orig": _metric_clean(nc0),

                    "psnr_attacked": _metric_clean(psnr1),
                    "ssim_attacked": _metric_clean(ssim1),
                    "nc_attacked": _metric_clean(nc1),
                    "nc_degradation": (
                        None if (_metric_clean(nc0) is None or _metric_clean(nc1) is None)
                        else float(_metric_clean(nc0) - _metric_clean(nc1))
                    ),

                    "time_sec": float(dt),
                    "watermarked_path": wm_img_path,
                    "flag_path": flag_path,
                    "attacked_path": attacked_path,
                    "extracted_path": extracted_path
                }

                if isinstance(extra_dbg, dict):
                    for k, v in extra_dbg.items():
                        row[f"dbg_{k}"] = v

                all_results.append(row)

            except Exception as e:
                print(f"    -> ERROR: {e}")
                traceback.print_exc()
                all_results.append({
                    "image_name": image_name,
                    "attack_type": attack_type,
                    "attack_params": _fmt_params(attack_params),

                    "psnr_orig": _metric_clean(psnr0),
                    "ssim_orig": _metric_clean(ssim0),
                    "nc_orig": _metric_clean(nc0),

                    "psnr_attacked": None,
                    "ssim_attacked": None,
                    "nc_attacked": None,
                    "nc_degradation": None,
                    "time_sec": None,

                    "watermarked_path": wm_img_path,
                    "flag_path": flag_path,
                    "attacked_path": None,
                    "extracted_path": None,
                    "error": str(e)
                })

    if not all_results:
        print("\nNo results produced (all runs failed or no images).")
        return

    csv_path = os.path.join(base_dir, "summary.csv")
    json_path = os.path.join(base_dir, "summary.json")

    keys = sorted({k for r in all_results for k in r.keys()})
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in all_results:
            w.writerow(r)

    baseline_avg = {
        "count_images": len(baseline_by_image),
        "psnr_original_avg": _safe_mean([x.get("psnr_orig") for x in baseline_by_image]),
        "ssim_original_avg": _safe_mean([x.get("ssim_orig") for x in baseline_by_image]),
        "nc_original_avg": _safe_mean([x.get("nc_orig") for x in baseline_by_image]),
        "embed_time_avg": _safe_mean([x.get("embed_time") for x in baseline_by_image]),
        "extract_time_baseline_avg": _safe_mean([x.get("extract_time_baseline") for x in baseline_by_image]),
    }

    global_summary = {
        "total_images": len(images),
        "total_runs_attempted": len(images) * len(ATTACK_SUITE),
        "total_rows": len(all_results),
        "success_rows": sum(1 for r in all_results if r.get("psnr_attacked") is not None),

        "psnr_attacked_avg": _safe_mean([r.get("psnr_attacked") for r in all_results]),
        "ssim_attacked_avg": _safe_mean([r.get("ssim_attacked") for r in all_results]),
        "nc_attacked_avg": _safe_mean([r.get("nc_attacked") for r in all_results]),
        "nc_degradation_avg": _safe_mean([r.get("nc_degradation") for r in all_results]),
        "time_sec_avg": _safe_mean([r.get("time_sec") for r in all_results]),
    }

    by_attack: Dict[str, List[dict]] = {}
    for r in all_results:
        key = f"{r.get('attack_type')} | {r.get('attack_params')}"
        by_attack.setdefault(key, []).append(r)

    per_attack_averages = {}
    for k, rows in by_attack.items():
        per_attack_averages[k] = {
            "count": len(rows),
            "psnr_attacked_avg": _safe_mean([x.get("psnr_attacked") for x in rows]),
            "ssim_attacked_avg": _safe_mean([x.get("ssim_attacked") for x in rows]),
            "nc_attacked_avg": _safe_mean([x.get("nc_attacked") for x in rows]),
            "nc_degradation_avg": _safe_mean([x.get("nc_degradation") for x in rows]),
        }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "baseline_avg_per_image": baseline_avg,
                "global_attacked_avg": global_summary,
                "per_attack_averages": per_attack_averages,
                "results": all_results
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n" + "=" * 70)
    print("BASELINE AVG (avg over input images, embed once per image)")
    print(f"Images counted: {baseline_avg['count_images']}")
    print(f"PSNR(original) avg : {baseline_avg['psnr_original_avg']}")
    print(f"SSIM(original) avg : {baseline_avg['ssim_original_avg']}")
    print(f"NC  (original) avg : {baseline_avg['nc_original_avg']}")
    print("-" * 70)
    print("GLOBAL ATTACKED AVG (avg over ALL images × ALL attacks)")
    print(f"Success rows: {global_summary['success_rows']} / {global_summary['total_rows']}")
    print(f"PSNR(attacked) avg : {global_summary['psnr_attacked_avg']}")
    print(f"SSIM(attacked) avg : {global_summary['ssim_attacked_avg']}")
    print(f"NC  (attacked) avg : {global_summary['nc_attacked_avg']}")
    print(f"NC degradation avg  : {global_summary['nc_degradation_avg']}")
    print("=" * 70)
    print(f"Saved CSV : {csv_path}")
    print(f"Saved JSON: {json_path}")
    print("=" * 70)


def main():
    input_folder = "/content"
    watermark_path = "/content/Cc.logo.circle.png"
    run_attack_suite(input_folder, watermark_path, base_dir="su_new_attacks", arnold_iter=ARNOLD_ITER)


if __name__ == "__main__":
    main()

AUTO RUN: Hess_new attack suite for ALL BMP images (flag_path)
Input folder   : /content
Watermark path : /content/Cc.logo.circle.png  (prepared -> su_new_attacks/watermark_prepared_binary.png)
BMP images     : 6
Total runs     : 66
[EMBED] host='Girl.bmp'
[EMBED] original_size=512x512, working_size=512x512, channel=B
[EMBED] payload_bits=4096, total_blocks=65536, selected_blocks=61440
[EMBED] full_repeat=16, odd_repeat=15
[EMBED] repeat_factor=15.0000x
[EMBED] q_eff=8.000000, q_margin=0.026000, r_margin=1.950000
[EMBED] wm_bits_ones=2689, wm_bits_zeros=1407
[EMBED][blk=0] pos=(252,16) bit=1 | Q: score=393792403.942950, mse=0.000000, raw=0.392292, avg=0.397762, worst=0.391590, ok=True | R: score=-0.337635, mse=1.817301, raw=-0.467013, avg=-0.600259, worst=-1.000000, ok=False
[EMBED][blk=1] pos=(82,228) bit=1 | Q: score=330409025.972139, mse=0.000000, raw=0.326615, avg=0.339992, worst=0.325519, ok=True | R: score=-0.386616, mse=1.449844, raw=-0.396220, avg=-0.545591, worst=-0.993725, ok

In [ ]:
# -*- coding: utf-8 -*-

import os, time, math, json, csv, traceback
from typing import Tuple, Dict, Any, Optional, List, Union

import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim


WM_SIZE = globals().get("WM_SIZE", 64)
ARNOLD_ITER = globals().get("ARNOLD_ITER", 10)


def prepare_watermark_for_opencv(
    watermark_path_in: str,
    out_binary_png: str,
    wm_size: int,
    svg_raster_size: int = 512,
    bg: int = 255
) -> str:
    """
    Converts watermark input (supports .svg, RGBA PNG) into a clean binary PNG (0/255),
    grayscale, size (wm_size, wm_size), with alpha composited onto white background.

    If input is .svg, requires: pip install cairosvg
    """
    path = watermark_path_in
    ext = os.path.splitext(path)[1].lower()

    if ext == ".svg":
        try:
            import cairosvg
        except Exception as e:
            raise RuntimeError(
                "SVG watermark needs cairosvg. Install once: pip install cairosvg"
            ) from e

        tmp_png = os.path.splitext(out_binary_png)[0] + "_raster.png"
        cairosvg.svg2png(
            url=path,
            write_to=tmp_png,
            output_width=int(svg_raster_size),
            output_height=int(svg_raster_size),
        )
        path = tmp_png

    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot read watermark: {watermark_path_in} (after raster: {path})")

    if img.ndim == 3 and img.shape[2] == 4:
        bgr = img[:, :, :3].astype(np.float32)
        a = img[:, :, 3:4].astype(np.float32) / 255.0
        bg_bgr = np.full_like(bgr, float(bg), dtype=np.float32)
        comp = bgr * a + bg_bgr * (1.0 - a)
        gray = cv2.cvtColor(comp.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    elif img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    gray = cv2.resize(gray, (wm_size, wm_size), interpolation=cv2.INTER_AREA)
    _, bin_wm = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

    out_dir = os.path.dirname(out_binary_png)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    cv2.imwrite(out_binary_png, bin_wm)
    return out_binary_png


def _binarize01(img: np.ndarray) -> np.ndarray:
    if img is None:
        return None
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return (img.astype(np.uint8) >= 128).astype(np.uint8)


def align_extracted_to_gt_binary(extracted_wm: np.ndarray, wm_gt_bin: np.ndarray) -> np.ndarray:
    if extracted_wm is None or wm_gt_bin is None:
        return extracted_wm

    gt01 = _binarize01(wm_gt_bin)
    ex01 = _binarize01(extracted_wm)

    if ex01.shape != gt01.shape:
        ex01 = cv2.resize(ex01, (gt01.shape[1], gt01.shape[0]), interpolation=cv2.INTER_NEAREST)

    same = float(np.mean(ex01 == gt01))
    inv = float(np.mean((1 - ex01) == gt01))

    if inv > same:
        ex01 = 1 - ex01

    return (ex01 * 255).astype(np.uint8)


def psnr_uint8(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float64)
    b = b.astype(np.float64)
    mse = np.mean((a - b) ** 2)
    if mse <= 1e-12:
        return 99.0
    return 10.0 * math.log10((255.0 ** 2) / mse)


def ssim_uint8(a: np.ndarray, b: np.ndarray) -> float:
    if a.ndim == 2:
        return float(ssim(a, b, data_range=255))
    try:
        return float(ssim(a, b, data_range=255, channel_axis=-1))
    except TypeError:
        return float(ssim(a, b, data_range=255, multichannel=True))


def nc_binary(wm_gt: np.ndarray, wm_ex: np.ndarray) -> float:
    if wm_gt is None or wm_ex is None:
        return float("nan")

    gt01 = _binarize01(wm_gt).astype(np.float64).ravel()
    ex01 = _binarize01(wm_ex).astype(np.float64)

    if wm_ex.ndim == 3:
        wm_ex_gray = cv2.cvtColor(wm_ex, cv2.COLOR_BGR2GRAY)
    else:
        wm_ex_gray = wm_ex

    if wm_gt.ndim == 3:
        wm_gt_gray = cv2.cvtColor(wm_gt, cv2.COLOR_BGR2GRAY)
    else:
        wm_gt_gray = wm_gt

    if wm_ex_gray.shape != wm_gt_gray.shape:
        ex01 = cv2.resize(
            ex01.astype(np.uint8),
            (wm_gt_gray.shape[1], wm_gt_gray.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.float64)
    ex01 = ex01.ravel()

    num = float(np.sum(gt01 * ex01))
    den = float(np.sqrt(np.sum(gt01 * gt01) * np.sum(ex01 * ex01)))
    nc1 = (num / den) if den > 1e-12 else (1.0 if np.array_equal(gt01, ex01) else 0.0)

    exi = 1.0 - ex01
    num2 = float(np.sum(gt01 * exi))
    den2 = float(np.sqrt(np.sum(gt01 * gt01) * np.sum(exi * exi)))
    nc2 = (num2 / den2) if den2 > 1e-12 else (1.0 if np.array_equal(gt01, exi) else 0.0)

    return max(nc1, nc2)


def calculate_metrics(host_img: np.ndarray, attacked_img: np.ndarray, extracted_wm: np.ndarray, wm_gt_bin: np.ndarray):
    p = psnr_uint8(host_img, attacked_img)
    s = ssim_uint8(host_img, attacked_img)
    n = nc_binary(wm_gt_bin, extracted_wm)
    return p, s, n


def _attack_dir_name(attack_type: str) -> str:
    return (
        attack_type.lower()
        .replace(" ", "_")
        .replace("&", "and")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "_")
        .replace(".", "p")
    )


def ensure_uint8(img):
    if img is None:
        return None
    if img.dtype == np.uint8:
        return img
    img = np.clip(img, 0, 255)
    return img.astype(np.uint8)


def _fmt_num(x) -> str:
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        s = f"{float(x):.6f}".rstrip("0").rstrip(".")
        if s == "-0":
            s = "0"
        return s.replace(".", "p").replace("-", "m")
    return str(x)


def _fmt_params(p) -> str:
    if p is None:
        return "None"
    if isinstance(p, tuple):
        return "(" + ",".join(_fmt_params(x) for x in p) + ")"
    if isinstance(p, list):
        return "[" + ",".join(_fmt_params(x) for x in p) + "]"
    if isinstance(p, float):
        return f"{p:.6f}".rstrip("0").rstrip(".")
    return str(p)


def _metric_clean(x):
    if x is None:
        return None
    try:
        v = float(x)
    except:
        return None
    if math.isnan(v):
        return None
    if math.isinf(v):
        return 99.0
    return v


def _safe_mean(vals: List[Any]) -> Optional[float]:
    buf = []
    for v in vals:
        vv = _metric_clean(v)
        if vv is None:
            continue
        buf.append(vv)
    if not buf:
        return None
    return float(sum(buf) / len(buf))


def scale_attack(image: np.ndarray, scale_factor: float) -> np.ndarray:
    if scale_factor <= 0:
        raise ValueError("scale_factor must be > 0")
    h, w = image.shape[:2]
    new_w = max(1, int(round(w * scale_factor)))
    new_h = max(1, int(round(h * scale_factor)))
    interp1 = cv2.INTER_AREA if scale_factor < 1.0 else cv2.INTER_CUBIC
    tmp = cv2.resize(image, (new_w, new_h), interpolation=interp1)
    interp2 = cv2.INTER_CUBIC if scale_factor < 1.0 else cv2.INTER_AREA
    out = cv2.resize(tmp, (w, h), interpolation=interp2)
    return out.astype(np.uint8)


def salt_pepper_attack(image: np.ndarray, noise_density: float, s_vs_p: float = 0.5, seed: int = 0) -> np.ndarray:
    if not (0.0 <= noise_density <= 1.0):
        raise ValueError("noise_density must be in [0,1]")
    if not (0.0 <= s_vs_p <= 1.0):
        raise ValueError("s_vs_p must be in [0,1]")
    rng = np.random.default_rng(seed)
    out = image.copy()
    h, w = out.shape[:2]
    num = int(round(noise_density * h * w))
    num_salt = int(round(num * s_vs_p))
    num_pepper = num - num_salt
    ys = rng.integers(0, h, size=num_salt, endpoint=False)
    xs = rng.integers(0, w, size=num_salt, endpoint=False)
    yp = rng.integers(0, h, size=num_pepper, endpoint=False)
    xp = rng.integers(0, w, size=num_pepper, endpoint=False)
    if out.ndim == 2:
        out[ys, xs] = 255
        out[yp, xp] = 0
    else:
        out[ys, xs, :] = 255
        out[yp, xp, :] = 0
    return out.astype(np.uint8)


def gaussian_noise_attack(image: np.ndarray, mean: float, variance: float, domain: str = "norm", seed: int = 0) -> np.ndarray:
    if variance < 0:
        raise ValueError("variance must be >= 0")
    rng = np.random.default_rng(seed)
    sigma = math.sqrt(variance)
    if domain == "pixel":
        img_f = image.astype(np.float32)
        noise = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out = np.clip(img_f + noise, 0.0, 255.0).astype(np.uint8)
        return out
    if domain == "norm":
        img_n = image.astype(np.float32) / 255.0
        noise = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out_n = np.clip(img_n + noise, 0.0, 1.0)
        out = (out_n * 255.0 + 0.5).astype(np.uint8)
        return out
    raise ValueError("domain must be 'norm' or 'pixel'")


def speckle_noise_attack(image: np.ndarray, mean: float, variance: float, domain: str = "norm", seed: int = 0) -> np.ndarray:
    if variance < 0:
        raise ValueError("variance must be >= 0")
    rng = np.random.default_rng(seed)
    sigma = math.sqrt(variance)
    if domain == "pixel":
        img_f = image.astype(np.float32)
        n = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out = np.clip(img_f + img_f * n, 0.0, 255.0)
        return out.astype(np.uint8)
    if domain == "norm":
        img_n = image.astype(np.float32) / 255.0
        n = rng.normal(loc=mean, scale=sigma, size=image.shape).astype(np.float32)
        out_n = np.clip(img_n + img_n * n, 0.0, 1.0)
        out = (out_n * 255.0 + 0.5).astype(np.uint8)
        return out
    raise ValueError("domain must be 'norm' or 'pixel'")


def blur_attack(image: np.ndarray, sigma: float) -> np.ndarray:
    if sigma < 0:
        raise ValueError("sigma must be >= 0")
    if sigma == 0:
        return image.copy()
    return cv2.GaussianBlur(image, (0, 0), sigmaX=sigma, sigmaY=sigma).astype(np.uint8)


def average_filter_attack(image: np.ndarray, ksize: int) -> np.ndarray:
    if ksize < 1:
        raise ValueError("ksize must be >= 1")
    return cv2.blur(image, (ksize, ksize)).astype(np.uint8)


def median_filter_attack(image: np.ndarray, ksize: int) -> np.ndarray:
    if ksize < 3 or (ksize % 2 == 0):
        raise ValueError("median ksize must be odd and >= 3 (e.g., 3,5,7)")
    return cv2.medianBlur(image, ksize).astype(np.uint8)


def histogram_attack(image: np.ndarray) -> np.ndarray:
    if image.ndim == 2:
        return cv2.equalizeHist(image).astype(np.uint8)
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
    out = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return out.astype(np.uint8)


def sharpen_attack(image: np.ndarray, sigma: float, amount: float = 1.5) -> np.ndarray:
    if sigma <= 0:
        return image.copy()
    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=sigma, sigmaY=sigma)
    out = cv2.addWeighted(image, 1.0 + amount, blurred, -amount, 0.0)
    return np.clip(out, 0, 255).astype(np.uint8)


def rotation_attack(
    image: np.ndarray,
    angle_deg: float,
    inverse_back: bool = True,
    border_mode: int = cv2.BORDER_CONSTANT,
    border_value: int = 0
) -> np.ndarray:
    h, w = image.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(
        image, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=border_mode,
        borderValue=border_value
    )
    if inverse_back:
        Minv = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)
        rotated = cv2.warpAffine(
            rotated, Minv, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=border_mode,
            borderValue=border_value
        )
    return rotated.astype(np.uint8)


def translation_attack(
    image: np.ndarray,
    shift_x: int,
    shift_y: int,
    inverse_back: bool = False,
    border_mode: int = cv2.BORDER_CONSTANT,
    border_value: int = 0
) -> np.ndarray:
    h, w = image.shape[:2]
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    shifted = cv2.warpAffine(
        image, M, (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=border_mode,
        borderValue=border_value
    )
    if inverse_back:
        Minv = np.float32([[1, 0, -shift_x], [0, 1, -shift_y]])
        shifted = cv2.warpAffine(
            shifted, Minv, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=border_mode,
            borderValue=border_value
        )
    return shifted.astype(np.uint8)


def jpeg_attack(image: np.ndarray, quality_factor: int) -> Tuple[np.ndarray, int]:
    q = int(np.clip(quality_factor, 0, 100))
    params = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    ok, enc = cv2.imencode(".jpg", image, params)
    if not ok:
        raise RuntimeError("cv2.imencode JPEG failed")
    flag = cv2.IMREAD_COLOR if image.ndim == 3 else cv2.IMREAD_GRAYSCALE
    out = cv2.imdecode(enc, flag)
    if out is None:
        raise RuntimeError("cv2.imdecode JPEG failed")
    return out.astype(np.uint8), int(enc.size)


def jp2_attack_ratio(
    image: np.ndarray,
    compression_ratio: float,
    force_lossy: bool = True,
    max_tries: int = 10
) -> Tuple[np.ndarray, Dict[str, Any]]:
    dbg: Dict[str, Any] = {}
    ratio_req = float(compression_ratio)
    if ratio_req < 1.0:
        ratio_req = 1.0
    dbg["compression_ratio_req"] = ratio_req
    dbg["raw_bytes"] = int(image.size)
    dbg["method"] = "opencv"
    dbg["force_lossy"] = bool(force_lossy)

    def enc_dec(rate_x1000: int) -> Tuple[np.ndarray, int]:
        params = [int(cv2.IMWRITE_JPEG2000_COMPRESSION_X1000), int(rate_x1000)]
        ok, enc = cv2.imencode(".jp2", image, params)
        if not ok:
            raise RuntimeError("JPEG2000 encode failed (OpenCV build may lack JP2 support).")
        flag = cv2.IMREAD_COLOR if image.ndim == 3 else cv2.IMREAD_GRAYSCALE
        out = cv2.imdecode(enc, flag)
        if out is None:
            raise RuntimeError("JPEG2000 decode failed.")
        return out.astype(np.uint8), int(enc.size)

    out_lossless, bytes_lossless = enc_dec(1000)
    dbg["lossless_test_rate_x1000"] = 1000
    dbg["lossless_jp2_bytes"] = bytes_lossless
    dbg["lossless_decoded_identical"] = bool(np.array_equal(out_lossless, image))

    rate = int(round(1000.0 / ratio_req))
    rate = max(1, min(rate, 1000))
    dbg["rate_x1000_initial"] = rate

    out, jp2_bytes = enc_dec(rate)
    dbg["jp2_bytes_initial"] = jp2_bytes
    dbg["jp2_identical_initial"] = bool(np.array_equal(out, image))
    dbg["psnr_initial(dB)"] = psnr_uint8(image, out)

    if not force_lossy:
        dbg["rate_x1000_final"] = rate
        dbg["jp2_bytes_final"] = jp2_bytes
        dbg["jp2_identical_final"] = bool(np.array_equal(out, image))
        dbg["psnr_final(dB)"] = psnr_uint8(image, out)
        return out, dbg

    tries = 0
    rate_cur = rate
    out_cur = out
    bytes_cur = jp2_bytes

    while tries < max_tries and np.array_equal(out_cur, image) and rate_cur > 1:
        tries += 1
        rate_cur = max(1, int(round(rate_cur * 0.75)))
        out_cur, bytes_cur = enc_dec(rate_cur)

    dbg["tries"] = tries
    dbg["rate_x1000_final"] = rate_cur
    dbg["jp2_bytes_final"] = bytes_cur
    dbg["jp2_identical_final"] = bool(np.array_equal(out_cur, image))
    dbg["psnr_final(dB)"] = psnr_uint8(image, out_cur)
    return out_cur, dbg


def crop_resize_attack_area(image: np.ndarray, crop_area_percent: float, where: str = "center") -> np.ndarray:
    if not (0.0 <= crop_area_percent < 100.0):
        raise ValueError("crop_area_percent must be in [0,100)")
    h, w = image.shape[:2]
    p = crop_area_percent / 100.0
    if p == 0.0:
        return image.copy()
    keep_area = 1.0 - p
    keep_side = math.sqrt(max(0.0, keep_area))
    rh = max(1, min(h, int(round(h * keep_side))))
    rw = max(1, min(w, int(round(w * keep_side))))

    def origin(H: int, W: int, rh_: int, rw_: int, where_: str) -> Tuple[int, int]:
        where_ = where_.lower()
        if where_ == "center":
            return (H - rh_) // 2, (W - rw_) // 2
        if where_ == "top":
            return 0, (W - rw_) // 2
        if where_ == "bottom":
            return H - rh_, (W - rw_) // 2
        if where_ == "left":
            return (H - rh_) // 2, 0
        if where_ == "right":
            return (H - rh_) // 2, W - rw_
        if where_ == "top-left":
            return 0, 0
        if where_ == "top-right":
            return 0, W - rw_
        if where_ == "bottom-left":
            return H - rh_, 0
        if where_ == "bottom-right":
            return H - rh_, W - rw_
        raise ValueError("unknown where")

    y0, x0 = origin(h, w, rh, rw, where)
    cropped = image[y0:y0 + rh, x0:x0 + rw]
    out = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)
    return out.astype(np.uint8)


OcclValue = Union[int, Tuple[int, int, int], str]


def occlusion_attack_area(
    image: np.ndarray,
    occlude_area_percent: float,
    where: str = "top-left",
    value: OcclValue = 0,
    seed: int = 0
) -> np.ndarray:
    if not (0.0 <= occlude_area_percent < 100.0):
        raise ValueError("occlude_area_percent must be in [0,100)")
    h, w = image.shape[:2]
    p = occlude_area_percent / 100.0
    if p == 0.0:
        return image.copy()

    side = math.sqrt(p)
    rh = max(1, min(h, int(round(h * side))))
    rw = max(1, min(w, int(round(w * side))))
    rng = np.random.default_rng(int(seed))

    def origin(H: int, W: int, rh_: int, rw_: int, where_: str) -> Tuple[int, int]:
        where_ = where_.lower()
        if where_ in {"random", "rand"}:
            y0 = int(rng.integers(0, max(1, H - rh_ + 1)))
            x0 = int(rng.integers(0, max(1, W - rw_ + 1)))
            return y0, x0
        if where_ == "center":
            return (H - rh_) // 2, (W - rw_) // 2
        if where_ == "top-left":
            return 0, 0
        if where_ == "top-right":
            return 0, W - rw_
        if where_ == "bottom-left":
            return H - rh_, 0
        if where_ == "bottom-right":
            return H - rh_, W - rw_
        if where_ == "top":
            return 0, (W - rw_) // 2
        if where_ == "bottom":
            return H - rh_, (W - rw_) // 2
        if where_ == "left":
            return (H - rh_) // 2, 0
        if where_ == "right":
            return (H - rh_) // 2, W - rw_
        raise ValueError("unknown where")

    out = image.copy()
    y0, x0 = origin(h, w, rh, rw, where)

    fill = value
    if isinstance(value, str):
        v = value.lower()
        if v == "mean":
            if out.ndim == 2:
                fill = int(np.round(float(np.mean(out))))
            else:
                m = np.mean(out.reshape(-1, 3), axis=0)
                fill = (int(round(m[0])), int(round(m[1])), int(round(m[2])))
        elif v == "random":
            if out.ndim == 2:
                patch = rng.integers(0, 256, size=(rh, rw), dtype=np.uint8)
                out[y0:y0 + rh, x0:x0 + rw] = patch
                return out.astype(np.uint8)
            else:
                patch = rng.integers(0, 256, size=(rh, rw, 3), dtype=np.uint8)
                out[y0:y0 + rh, x0:x0 + rw, :] = patch
                return out.astype(np.uint8)
        else:
            raise ValueError("value string must be 'mean' or 'random'")

    if out.ndim == 2:
        out[y0:y0 + rh, x0:x0 + rw] = int(fill)
    else:
        if isinstance(fill, (tuple, list)):
            if len(fill) != 3:
                raise ValueError("color fill tuple must be (B,G,R)")
            out[y0:y0 + rh, x0:x0 + rw, :] = np.array(fill, dtype=np.uint8)
        else:
            out[y0:y0 + rh, x0:x0 + rw, :] = int(fill)

    return out.astype(np.uint8)


def lowpass_filter_attack(image: np.ndarray, kernel_size_x: int, kernel_size_y: int) -> np.ndarray:
    if kernel_size_x <= 0 or kernel_size_y <= 0:
        raise ValueError("kernel sizes must be positive")
    if kernel_size_x % 2 == 0:
        kernel_size_x += 1
    if kernel_size_y % 2 == 0:
        kernel_size_y += 1
    return cv2.GaussianBlur(image, (kernel_size_x, kernel_size_y), 0).astype(np.uint8)


def gamma_attack(image: np.ndarray, gamma: float) -> np.ndarray:
    if gamma <= 0:
        raise ValueError("gamma must be > 0")
    inv = 1.0 / float(gamma)
    lut = np.array([((i / 255.0) ** inv) * 255.0 for i in range(256)], dtype=np.float32)
    lut = np.clip(lut + 0.5, 0, 255).astype(np.uint8)
    return cv2.LUT(image, lut).astype(np.uint8)


def motion_blur_attack(image: np.ndarray, ksize: int, direction: str = "h") -> np.ndarray:
    if ksize < 1:
        raise ValueError("ksize must be >= 1")
    ksize = int(ksize)
    if ksize % 2 == 0:
        ksize += 1
    kernel = np.zeros((ksize, ksize), dtype=np.float32)
    direction = direction.lower()
    if direction == "h":
        kernel[ksize // 2, :] = 1.0
    elif direction == "v":
        kernel[:, ksize // 2] = 1.0
    else:
        raise ValueError("direction must be 'h' or 'v'")
    kernel /= kernel.sum()
    out = cv2.filter2D(image, -1, kernel)
    return np.clip(out, 0, 255).astype(np.uint8)


def brightness_contrast_attack(image: np.ndarray, alpha: float = 1.0, beta: float = 0.0) -> np.ndarray:
    img_f = image.astype(np.float32)
    out = img_f * float(alpha) + float(beta)
    return np.clip(out + 0.5, 0, 255).astype(np.uint8)


def clahe_y_attack(image: np.ndarray, clip_limit: float = 2.0, tile_grid: int = 8) -> np.ndarray:
    if image.ndim == 2:
        clahe = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=(int(tile_grid), int(tile_grid)))
        return clahe.apply(image).astype(np.uint8)
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    clahe = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=(int(tile_grid), int(tile_grid)))
    ycrcb[:, :, 0] = clahe.apply(ycrcb[:, :, 0])
    out = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    return out.astype(np.uint8)


def bilateral_attack(image: np.ndarray, d: int = 7, sigma_color: float = 30.0, sigma_space: float = 7.0) -> np.ndarray:
    out = cv2.bilateralFilter(image, int(d), float(sigma_color), float(sigma_space))
    return np.clip(out, 0, 255).astype(np.uint8)


ATTACK_SUITE = [
    ("Blur", 1),
    ("Sharpen", (1, 1.5)),
    ("Gaussian Noise", (0.0, 0.003)),
    ("Salt & Pepper", 0.1),
    ("JPEG", 50),
    ("JPEG2000", 13),
    ("Lowpass", (9, 9)),
    ("Scale", 0.2),
    ("Rotation", 45),
    ("Gamma", 1.5),
    ("Occlusion", (50.0, "random", 0, 3)),
]


def apply_attack(wm_img: np.ndarray, attack_type: str, attack_param):
    extra_dbg = None

    if attack_type == "Blur":
        sigma = float(attack_param)
        out = blur_attack(wm_img, sigma)
        pstr = f"_sigma{_fmt_num(sigma)}"
        return out, pstr, extra_dbg

    if attack_type == "Sharpen":
        sigma, amount = attack_param if isinstance(attack_param, tuple) else (float(attack_param), 1.5)
        sigma = float(sigma)
        amount = float(amount)
        out = sharpen_attack(wm_img, sigma, amount=amount)
        pstr = f"_sigma{_fmt_num(sigma)}_a{_fmt_num(amount)}"
        return out, pstr, extra_dbg

    if attack_type == "Gaussian Noise":
        mean, var = attack_param
        out = gaussian_noise_attack(wm_img, float(mean), float(var), domain="norm")
        pstr = f"_m{_fmt_num(mean)}_v{_fmt_num(var)}"
        return out, pstr, extra_dbg

    if attack_type == "Speckle Noise":
        mean, var = attack_param
        out = speckle_noise_attack(wm_img, float(mean), float(var), domain="norm")
        pstr = f"_m{_fmt_num(mean)}_v{_fmt_num(var)}"
        return out, pstr, extra_dbg

    if attack_type == "Salt & Pepper":
        d = float(attack_param)
        out = salt_pepper_attack(wm_img, d)
        pstr = f"_d{_fmt_num(d)}"
        return out, pstr, extra_dbg

    if attack_type == "JPEG":
        qf = int(attack_param)
        out, jpeg_bytes = jpeg_attack(wm_img, qf)
        extra_dbg = {"jpeg_bytes": int(jpeg_bytes)}
        pstr = f"_qf{qf}"
        return out, pstr, extra_dbg

    if attack_type == "JPEG2000":
        ratio = float(attack_param)
        out, dbg = jp2_attack_ratio(wm_img, ratio, force_lossy=True)
        extra_dbg = dbg
        rate_final = dbg.get("rate_x1000_final", None)
        if rate_final is not None:
            pstr = f"_cr{_fmt_num(ratio)}_r{int(rate_final)}"
        else:
            pstr = f"_cr{_fmt_num(ratio)}"
        return out, pstr, extra_dbg

    if attack_type == "Median":
        k = int(attack_param)
        out = median_filter_attack(wm_img, k)
        pstr = f"_k{k}"
        return out, pstr, extra_dbg

    if attack_type == "Average":
        k = int(attack_param)
        out = average_filter_attack(wm_img, k)
        pstr = f"_k{k}"
        return out, pstr, extra_dbg

    if attack_type == "Lowpass":
        kx, ky = attack_param
        out = lowpass_filter_attack(wm_img, int(kx), int(ky))
        pstr = f"_kx{int(kx)}_ky{int(ky)}"
        return out, pstr, extra_dbg

    if attack_type == "Scale":
        s_ = float(attack_param)
        out = scale_attack(wm_img, s_)
        pstr = f"_s{_fmt_num(s_)}"
        return out, pstr, extra_dbg

    if attack_type == "Crop":
        crop_percent, where = attack_param
        crop_percent = float(crop_percent)
        where = str(where).lower()
        out = crop_resize_attack_area(wm_img, crop_percent, where=where)
        pstr = f"_p{_fmt_num(crop_percent)}_{where}"
        return out, pstr, extra_dbg

    if attack_type == "Rotation":
        ang = float(attack_param)
        out = rotation_attack(wm_img, ang, inverse_back=True)
        pstr = f"_a{_fmt_num(ang)}"
        return out, pstr, extra_dbg

    if attack_type == "Translation":
        sx, sy = attack_param
        out = translation_attack(wm_img, int(sx), int(sy), inverse_back=False)
        pstr = f"_dx{int(sx)}_dy{int(sy)}"
        return out, pstr, extra_dbg

    if attack_type == "Gamma":
        g = float(attack_param)
        out = gamma_attack(wm_img, g)
        pstr = f"_g{_fmt_num(g)}"
        return out, pstr, extra_dbg

    if attack_type == "Motion Blur":
        ksize, direction = attack_param
        out = motion_blur_attack(wm_img, int(ksize), str(direction).lower())
        pstr = f"_k{int(ksize)}_{str(direction).lower()}"
        return out, pstr, extra_dbg

    if attack_type == "BrightnessContrast":
        alpha, beta = attack_param
        out = brightness_contrast_attack(wm_img, float(alpha), float(beta))
        pstr = f"_a{_fmt_num(alpha)}_b{_fmt_num(beta)}"
        return out, pstr, extra_dbg

    if attack_type == "CLAHE_Y":
        clip, tile = attack_param
        out = clahe_y_attack(wm_img, float(clip), int(tile))
        pstr = f"_c{_fmt_num(clip)}_t{int(tile)}"
        return out, pstr, extra_dbg

    if attack_type == "Bilateral":
        d, sc, ss_ = attack_param
        out = bilateral_attack(wm_img, int(d), float(sc), float(ss_))
        pstr = f"_d{int(d)}_sc{_fmt_num(sc)}_ss{_fmt_num(ss_)}"
        return out, pstr, extra_dbg

    if attack_type == "Histogram":
        out = histogram_attack(wm_img)
        pstr = "_eq"
        return out, pstr, extra_dbg

    if attack_type == "Occlusion":
        if isinstance(attack_param, (tuple, list)):
            if len(attack_param) == 3:
                p, where, value = attack_param
                seed = 0
            elif len(attack_param) == 4:
                p, where, value, seed = attack_param
            else:
                raise ValueError("Occlusion params must be (p, where, value) or (p, where, value, seed)")
        else:
            raise ValueError("Occlusion params must be a tuple/list")

        out = occlusion_attack_area(
            wm_img,
            float(p),
            where=str(where).lower(),
            value=value,
            seed=int(seed)
        )
        pstr = f"_p{_fmt_num(p)}_{str(where).lower()}_v{str(value).lower() if isinstance(value, str) else value}_s{int(seed)}"
        return out, pstr, extra_dbg

    raise ValueError(f"Unknown attack_type: {attack_type}")


def run_attack_suite(input_folder: str, watermark_path: str, base_dir: str = "su_new_attacks", arnold_iter: int = ARNOLD_ITER):
    os.makedirs(base_dir, exist_ok=True)

    prepared_wm_path = os.path.join(base_dir, "watermark_prepared_binary.png")
    watermark_path_use = prepare_watermark_for_opencv(
        watermark_path_in=watermark_path,
        out_binary_png=prepared_wm_path,
        wm_size=WM_SIZE,
        svg_raster_size=512,
        bg=255
    )

    images = sorted([f for f in os.listdir(input_folder) if f.lower().endswith(".bmp")])
    if not images:
        raise FileNotFoundError(f"No .bmp images found in {input_folder}")

    wm_raw = cv2.imread(watermark_path_use, cv2.IMREAD_GRAYSCALE)
    if wm_raw is None:
        raise FileNotFoundError(f"Cannot read prepared watermark: {watermark_path_use}")
    wm_raw = cv2.resize(wm_raw, (WM_SIZE, WM_SIZE), interpolation=cv2.INTER_NEAREST)
    _, wm_gt_bin = cv2.threshold(wm_raw, 127, 255, cv2.THRESH_BINARY)

    watermarked_dir = os.path.join(base_dir, "watermarked_once")
    attacked_root = os.path.join(base_dir, "attacked")
    extracted_root = os.path.join(base_dir, "extracted")
    os.makedirs(watermarked_dir, exist_ok=True)
    os.makedirs(attacked_root, exist_ok=True)
    os.makedirs(extracted_root, exist_ok=True)

    total_jobs = len(images) * len(ATTACK_SUITE)
    job_idx = 0

    print("=" * 70)
    print("AUTO RUN: Hess_new attack suite for ALL BMP images (flag_path)")
    print(f"Input folder   : {input_folder}")
    print(f"Watermark path : {watermark_path}  (prepared -> {watermark_path_use})")
    print(f"BMP images     : {len(images)}")
    print(f"Total runs     : {total_jobs}")
    print("=" * 70)

    all_results = []
    baseline_by_image = []

    for img_i, image_name in enumerate(images, 1):
        host_path = os.path.join(input_folder, image_name)
        host_img = cv2.imread(host_path, cv2.IMREAD_COLOR)
        if host_img is None:
            print(f"[SKIP] cannot read: {image_name}")
            continue

        base_name = os.path.splitext(image_name)[0]

        wm_img_path = os.path.join(watermarked_dir, f"watermarked_{base_name}.png")
        flag_path = os.path.join(watermarked_dir, f"flag_{base_name}.txt")
        wm_extract0 = os.path.join(watermarked_dir, f"extracted_baseline_{base_name}.png")

        try:
            t0 = time.time()
            wm_img = embed(host_path, watermark_path_use, wm_img_path, flag_path, arnold_iter=arnold_iter)
            wm_img = ensure_uint8(wm_img)
            t_embed = time.time() - t0

            t1 = time.time()
            ext0 = extract(wm_img_path, flag_path, wm_extract0, (WM_SIZE, WM_SIZE), arnold_iter=arnold_iter)
            ext0 = ensure_uint8(ext0)
            ext0 = align_extracted_to_gt_binary(ext0, wm_gt_bin)
            cv2.imwrite(wm_extract0, ext0)
            t_extract0 = time.time() - t1

            psnr0, ssim0, nc0 = calculate_metrics(host_img, wm_img, ext0, wm_gt_bin)
            baseline_by_image.append({
                "image": image_name,
                "psnr_orig": psnr0,
                "ssim_orig": ssim0,
                "nc_orig": nc0,
                "embed_time": t_embed,
                "extract_time_baseline": t_extract0
            })

            print(
                f"\n[{img_i}/{len(images)}] BASELINE {image_name}: "
                f"PSNR={_metric_clean(psnr0)} SSIM={_metric_clean(ssim0)} NC={_metric_clean(nc0)} "
                f"(embed {t_embed:.2f}s, extract {t_extract0:.2f}s)"
            )

        except Exception as e:
            print(f"[FAIL] baseline embed/extract for {image_name}: {e}")
            traceback.print_exc()
            continue

        for attack_type, attack_params in ATTACK_SUITE:
            job_idx += 1
            print(f"  [{job_idx}/{total_jobs}] {image_name} | {attack_type} | {attack_params}")

            atk_dir = os.path.join(attacked_root, _attack_dir_name(attack_type))
            ext_dir = os.path.join(extracted_root, _attack_dir_name(attack_type))
            os.makedirs(atk_dir, exist_ok=True)
            os.makedirs(ext_dir, exist_ok=True)

            try:
                tA = time.time()
                attacked_img, param_str, extra_dbg = apply_attack(wm_img, attack_type, attack_params)
                attacked_img = ensure_uint8(attacked_img)

                attacked_path = os.path.join(atk_dir, f"{_attack_dir_name(attack_type)}_{base_name}{param_str}.bmp")
                cv2.imwrite(attacked_path, attacked_img)

                extracted_path = os.path.join(ext_dir, f"extracted_{base_name}{param_str}.png")
                ext_wm = extract(attacked_path, flag_path, extracted_path, (WM_SIZE, WM_SIZE), arnold_iter=arnold_iter)
                ext_wm = ensure_uint8(ext_wm)
                ext_wm = align_extracted_to_gt_binary(ext_wm, wm_gt_bin)
                cv2.imwrite(extracted_path, ext_wm)

                psnr1, ssim1, nc1 = calculate_metrics(host_img, attacked_img, ext_wm, wm_gt_bin)
                dt = time.time() - tA

                row = {
                    "image_name": image_name,
                    "attack_type": attack_type,
                    "attack_params": _fmt_params(attack_params),

                    "psnr_orig": _metric_clean(psnr0),
                    "ssim_orig": _metric_clean(ssim0),
                    "nc_orig": _metric_clean(nc0),

                    "psnr_attacked": _metric_clean(psnr1),
                    "ssim_attacked": _metric_clean(ssim1),
                    "nc_attacked": _metric_clean(nc1),
                    "nc_degradation": (
                        None if (_metric_clean(nc0) is None or _metric_clean(nc1) is None)
                        else float(_metric_clean(nc0) - _metric_clean(nc1))
                    ),

                    "time_sec": float(dt),
                    "watermarked_path": wm_img_path,
                    "flag_path": flag_path,
                    "attacked_path": attacked_path,
                    "extracted_path": extracted_path
                }

                if isinstance(extra_dbg, dict):
                    for k, v in extra_dbg.items():
                        row[f"dbg_{k}"] = v

                all_results.append(row)

            except Exception as e:
                print(f"    -> ERROR: {e}")
                traceback.print_exc()
                all_results.append({
                    "image_name": image_name,
                    "attack_type": attack_type,
                    "attack_params": _fmt_params(attack_params),

                    "psnr_orig": _metric_clean(psnr0),
                    "ssim_orig": _metric_clean(ssim0),
                    "nc_orig": _metric_clean(nc0),

                    "psnr_attacked": None,
                    "ssim_attacked": None,
                    "nc_attacked": None,
                    "nc_degradation": None,
                    "time_sec": None,

                    "watermarked_path": wm_img_path,
                    "flag_path": flag_path,
                    "attacked_path": None,
                    "extracted_path": None,
                    "error": str(e)
                })

    if not all_results:
        print("\nNo results produced (all runs failed or no images).")
        return

    csv_path = os.path.join(base_dir, "summary.csv")
    json_path = os.path.join(base_dir, "summary.json")

    keys = sorted({k for r in all_results for k in r.keys()})
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in all_results:
            w.writerow(r)

    baseline_avg = {
        "count_images": len(baseline_by_image),
        "psnr_original_avg": _safe_mean([x.get("psnr_orig") for x in baseline_by_image]),
        "ssim_original_avg": _safe_mean([x.get("ssim_orig") for x in baseline_by_image]),
        "nc_original_avg": _safe_mean([x.get("nc_orig") for x in baseline_by_image]),
        "embed_time_avg": _safe_mean([x.get("embed_time") for x in baseline_by_image]),
        "extract_time_baseline_avg": _safe_mean([x.get("extract_time_baseline") for x in baseline_by_image]),
    }

    global_summary = {
        "total_images": len(images),
        "total_runs_attempted": len(images) * len(ATTACK_SUITE),
        "total_rows": len(all_results),
        "success_rows": sum(1 for r in all_results if r.get("psnr_attacked") is not None),

        "psnr_attacked_avg": _safe_mean([r.get("psnr_attacked") for r in all_results]),
        "ssim_attacked_avg": _safe_mean([r.get("ssim_attacked") for r in all_results]),
        "nc_attacked_avg": _safe_mean([r.get("nc_attacked") for r in all_results]),
        "nc_degradation_avg": _safe_mean([r.get("nc_degradation") for r in all_results]),
        "time_sec_avg": _safe_mean([r.get("time_sec") for r in all_results]),
    }

    by_attack: Dict[str, List[dict]] = {}
    for r in all_results:
        key = f"{r.get('attack_type')} | {r.get('attack_params')}"
        by_attack.setdefault(key, []).append(r)

    per_attack_averages = {}
    for k, rows in by_attack.items():
        per_attack_averages[k] = {
            "count": len(rows),
            "psnr_attacked_avg": _safe_mean([x.get("psnr_attacked") for x in rows]),
            "ssim_attacked_avg": _safe_mean([x.get("ssim_attacked") for x in rows]),
            "nc_attacked_avg": _safe_mean([x.get("nc_attacked") for x in rows]),
            "nc_degradation_avg": _safe_mean([x.get("nc_degradation") for x in rows]),
        }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "baseline_avg_per_image": baseline_avg,
                "global_attacked_avg": global_summary,
                "per_attack_averages": per_attack_averages,
                "results": all_results
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n" + "=" * 70)
    print("BASELINE AVG (avg over input images, embed once per image)")
    print(f"Images counted: {baseline_avg['count_images']}")
    print(f"PSNR(original) avg : {baseline_avg['psnr_original_avg']}")
    print(f"SSIM(original) avg : {baseline_avg['ssim_original_avg']}")
    print(f"NC  (original) avg : {baseline_avg['nc_original_avg']}")
    print("-" * 70)
    print("GLOBAL ATTACKED AVG (avg over ALL images × ALL attacks)")
    print(f"Success rows: {global_summary['success_rows']} / {global_summary['total_rows']}")
    print(f"PSNR(attacked) avg : {global_summary['psnr_attacked_avg']}")
    print(f"SSIM(attacked) avg : {global_summary['ssim_attacked_avg']}")
    print(f"NC  (attacked) avg : {global_summary['nc_attacked_avg']}")
    print(f"NC degradation avg  : {global_summary['nc_degradation_avg']}")
    print("=" * 70)
    print(f"Saved CSV : {csv_path}")
    print(f"Saved JSON: {json_path}")
    print("=" * 70)


def main():
    input_folder = "/content"
    watermark_path = "/content/wm.png"
    run_attack_suite(input_folder, watermark_path, base_dir="su_new_attacks", arnold_iter=ARNOLD_ITER)


if __name__ == "__main__":
    main()

AUTO RUN: Hess_new attack suite for ALL BMP images (flag_path)
Input folder   : /content
Watermark path : /content/wm.png  (prepared -> su_new_attacks/watermark_prepared_binary.png)
BMP images     : 6
Total runs     : 66
[EMBED] host='Girl.bmp'
[EMBED] original_size=512x512, working_size=512x512, channel=B
[EMBED] payload_bits=4096, total_blocks=65536, selected_blocks=61440
[EMBED] full_repeat=16, odd_repeat=15
[EMBED] repeat_factor=15.0000x
[EMBED] q_eff=8.000000, q_margin=0.026000, r_margin=1.950000
[EMBED] wm_bits_ones=2059, wm_bits_zeros=2037
[EMBED][blk=0] pos=(252,16) bit=1 | Q: score=393792403.942950, mse=0.000000, raw=0.392292, avg=0.397762, worst=0.391590, ok=True | R: score=-0.337635, mse=1.817301, raw=-0.467013, avg=-0.600259, worst=-1.000000, ok=False
[EMBED][blk=1] pos=(82,228) bit=0 | Q: score=-0.000151, mse=337.025402, raw=0.031600, avg=-0.043355, worst=-0.268367, ok=False | R: score=0.912995, mse=0.633465, raw=0.545039, avg=0.656681, worst=0.544134, ok=True
[EMBED][blk=